# Model 1: Value-only     | Quarterly rebalancing | 3 portfolios     
# Model 2: Momentum-only  | Weekly rebalancing    | 3 portfolios     
                                                                      
# Portfolio construction:                                             
    # Within each sector → sort by signal → assign top/mid/bot third   
    # Combine across sectors → P1 / P2 / P3                            
                                                                      
Vietnamese Market Adaptations:                                      
    1. Long-only (no shorting)                                        
    2. Sector-based value: P/B for Banks/Financials, P/E for others  
    3. Negative P/E excluded from ranking                             
    4. 45-day publication lag on quarterly data                       
    5. Trading fee model for weekly momentum rebalancing            

In [1]:
# ── CELL 1: IMPORT LIBRARIES ──────────────────────────────────────────

import pandas as pd
import numpy as np
import re
import io
import warnings
from google.colab import files

warnings.filterwarnings("ignore")

print("✓ Libraries loaded")

✓ Libraries loaded


In [2]:
# ── CELL 2: CONFIGURATION ─────────────────────────────────────────────
# --> EDIT THIS CELL to match your data

# Price file column names
COL_DATE    = "time"
COL_TICKER  = "ticker"
COL_CLOSE   = "close"
COL_VOLUME  = "volume"
DATE_FORMAT = "%m/%d/%Y %H:%M"

# Value signal settings
FINANCIAL_SECTORS = ["Financials", "Banks"]   # use P/B for these sectors
PE_MIN            = 0                         # exclude P/E below this
PE_MAX            = 200                       # exclude P/E above this
PUB_LAG_DAYS      = 45                        # days after quarter end before data is safe

# Momentum settings
MOM_WEEK_START    = 0     # skip last X weeks
MOM_WEEK_END      = 1    # look back Y weeks total

# Portfolio settings
MIN_SECTOR_SIZE   = 6     # sectors below this → all stocks go to P2

# Trading fee settings (Vietnam market)
BUY_FEE_PCT       = 0.20  # brokerage fee on buy (%)
SELL_FEE_PCT      = 0.20  # brokerage fee on sell (%)
SELL_TAX_PCT      = 0.10  # personal income tax on sell (%)

print("✓ Configuration set")
print(f"  Financial sectors   : {FINANCIAL_SECTORS} → use 1/P/B")
print(f"  Other sectors       : use 1/P/E")
print(f"  Min sector size     : {MIN_SECTOR_SIZE} stocks")
print(f"  Round-trip cost     : {BUY_FEE_PCT + SELL_FEE_PCT + SELL_TAX_PCT:.2f}%")

✓ Configuration set
  Financial sectors   : ['Financials', 'Banks'] → use 1/P/B
  Other sectors       : use 1/P/E
  Min sector size     : 6 stocks
  Round-trip cost     : 0.50%


In [3]:
# ── CELL 3: UPLOAD PRICE FILES ────────────────────────────────────────
# Click "Choose Files" and select all 5 CSV price files at once
# (hold Ctrl to select multiple files)

print("Upload all 5 price CSV files...")
uploaded_prices = files.upload()

print(f"\n✓ {len(uploaded_prices)} files uploaded:")
for fname in uploaded_prices:
    print(f"  {fname}")

Upload all 5 price CSV files...


Saving vn_stocks_daily_batch2.csv to vn_stocks_daily_batch2.csv
Saving vn_stocks_daily_batch3.csv to vn_stocks_daily_batch3.csv
Saving vn_stocks_daily_batch4.csv to vn_stocks_daily_batch4.csv
Saving vn_stocks_daily_batch5.csv to vn_stocks_daily_batch5.csv
Saving vn_stocks_daily_batch6.csv to vn_stocks_daily_batch6.csv

✓ 5 files uploaded:
  vn_stocks_daily_batch2.csv
  vn_stocks_daily_batch3.csv
  vn_stocks_daily_batch4.csv
  vn_stocks_daily_batch5.csv
  vn_stocks_daily_batch6.csv


In [29]:
# ── CELL 4: UPLOAD FUNDAMENTAL DATA FILE ─────────────────────────────
# Upload the cleaned FiinProX Excel file
# (FiinProX_DE_Corporate_cleaned.xlsx — with Q2-Q4 2026 removed)

print("Upload the cleaned FiinProX Excel file...")
uploaded_fund = files.upload()

fund_filename = list(uploaded_fund.keys())[0]
print(f"\n✓ Uploaded: {fund_filename}")

Upload the cleaned FiinProX Excel file...


Saving FiinProX_DE_Corporate_20260424 (1).xlsx to FiinProX_DE_Corporate_20260424 (1) (1).xlsx

✓ Uploaded: FiinProX_DE_Corporate_20260424 (1) (1).xlsx


In [30]:
# ── CELL 5: LOAD AND COMBINE PRICE FILES ─────────────────────────────

print("=" * 60)
print("LOADING PRICE DATA")
print("=" * 60)

frames = []
for fname, content in uploaded_prices.items():
    df = pd.read_csv(io.BytesIO(content))
    print(f"  {fname}: {len(df):>10,} rows | {df[COL_TICKER].nunique():>4} stocks")
    frames.append(df)

prices_raw = pd.concat(frames, ignore_index=True)

# Standardize column names
prices_raw = prices_raw.rename(columns={
    COL_DATE   : "date",
    COL_TICKER : "ticker",
    COL_CLOSE  : "close",
    COL_VOLUME : "volume",
})

# Parse and clean
prices_raw["date"]   = pd.to_datetime(prices_raw["date"], format=DATE_FORMAT)
prices_raw["ticker"] = prices_raw["ticker"].str.strip().str.upper()
prices_raw["close"]  = pd.to_numeric(prices_raw["close"],  errors="coerce")
prices_raw["volume"] = pd.to_numeric(prices_raw["volume"], errors="coerce")

# Remove duplicates and bad prices
prices_raw = prices_raw.drop_duplicates(subset=["date", "ticker"], keep="first")
prices_raw = prices_raw[prices_raw["close"] > 0].dropna(subset=["close"])
prices_raw = prices_raw.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"\n  Combined : {len(prices_raw):,} rows | {prices_raw['ticker'].nunique()} stocks")
print(f"  Date range: {prices_raw['date'].min().date()} → {prices_raw['date'].max().date()}")
print("✓ Price data loaded")

LOADING PRICE DATA
  vn_stocks_daily_batch2.csv:    138,176 rows |   60 stocks
  vn_stocks_daily_batch3.csv:    138,450 rows |   60 stocks
  vn_stocks_daily_batch4.csv:    143,401 rows |   60 stocks
  vn_stocks_daily_batch5.csv:    141,708 rows |   61 stocks
  vn_stocks_daily_batch6.csv:    139,145 rows |   60 stocks

  Combined : 700,880 rows | 301 stocks
  Date range: 2015-09-28 → 2026-05-18
✓ Price data loaded


In [31]:
# ── CELL 6: BUILD PRICE MATRICES ─────────────────────────────────────

print("Building price matrices...")

# Wide format daily matrix
daily_wide = prices_raw.pivot(index="date", columns="ticker", values="close")

# Monthly (for reference)
monthly_prices  = daily_wide.resample("ME").last()
monthly_returns = monthly_prices.pct_change().iloc[1:]

# Weekly (for momentum model)
# FIXED — anchors to Friday (W-FRI)
weekly_prices  = daily_wide.resample("W-THU").last()
weekly_returns = weekly_prices.pct_change().iloc[1:]

print(f"  Daily  matrix : {daily_wide.shape}")
print(f"  Monthly matrix: {monthly_prices.shape}")
print(f"  Weekly  matrix: {weekly_prices.shape}")
print("✓ Price matrices built")

Building price matrices...
  Daily  matrix : (2657, 301)
  Monthly matrix: (129, 301)
  Weekly  matrix: (556, 301)
✓ Price matrices built


In [32]:
# ── CELL 7: LOAD AND PARSE FUNDAMENTAL DATA ───────────────────────────

print("Parsing fundamental data...")

df_raw = pd.read_excel(
    io.BytesIO(uploaded_fund[fund_filename]),
    sheet_name="Sheet1"
)

df_raw.columns = [str(c).replace("\n", " ").strip() for c in df_raw.columns]

SECTOR_COL = "Industrial sector (ICB) L1"

# Check required columns
required_cols = ["Ticker", SECTOR_COL]
missing_cols = [c for c in required_cols if c not in df_raw.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Identify signal columns
pe_cols = [c for c in df_raw.columns if "P/E" in c and "Quarter" in c]
pb_cols = [c for c in df_raw.columns if "P/B" in c and "Quarter" in c]

print(f"  P/E columns : {len(pe_cols)}")
print(f"  P/B columns : {len(pb_cols)}")

if len(pe_cols) == 0:
    raise ValueError("No P/E columns found. Check column names.")

if len(pb_cols) == 0:
    raise ValueError("No P/B columns found. Check column names.")

def extract_quarter(col):
    m = re.search(r"Q\d\.\d{4}", col)
    return m.group(0) if m else None

def quarter_to_safe_date(q_str, lag=PUB_LAG_DAYS):
    qe = {
        "Q1": "03-31",
        "Q2": "06-30",
        "Q3": "09-30",
        "Q4": "12-31"
    }
    q, year = q_str.split(".")
    return pd.to_datetime(f"{year}-{qe[q]}") + pd.Timedelta(days=lag)

def melt_metric(df, cols, col_label, id_cols):
    melted = df[id_cols + cols].melt(
        id_vars=id_cols,
        value_vars=cols,
        var_name="col",
        value_name=col_label
    )
    melted["quarter"] = melted["col"].apply(extract_quarter)
    melted = melted.drop(columns="col")
    melted = melted.dropna(subset=["quarter"])
    return melted

df_pe = melt_metric(df_raw, pe_cols, "pe", ["Ticker", SECTOR_COL])
df_pb = melt_metric(df_raw, pb_cols, "pb", ["Ticker"])

fund = df_pe.merge(
    df_pb,
    on=["Ticker", "quarter"],
    how="inner"
)

fund = fund.rename(columns={
    "Ticker": "ticker",
    SECTOR_COL: "sector"
})

fund["ticker"] = fund["ticker"].astype(str).str.strip().str.upper()
fund["sector"] = fund["sector"].astype(str).str.strip()

fund["pe"] = pd.to_numeric(fund["pe"], errors="coerce")
fund["pb"] = pd.to_numeric(fund["pb"], errors="coerce")

fund["safe_date"] = fund["quarter"].apply(quarter_to_safe_date)

fund = fund.sort_values(["ticker", "safe_date"]).reset_index(drop=True)

# Sector lookup: most recent sector per ticker
sector_lookup = (
    fund.sort_values("safe_date")
    .drop_duplicates("ticker", keep="last")
    .set_index("ticker")["sector"]
)

print(f"\n  Stocks     : {fund['ticker'].nunique()}")
print(f"  Quarters   : {fund['quarter'].nunique()}")
print(f"\n  Sector breakdown:")
print(fund.drop_duplicates("ticker")["sector"].value_counts().to_string())
print("✓ Fundamental data loaded")

Parsing fundamental data...
  P/E columns : 21
  P/B columns : 21

  Stocks     : 301
  Quarters   : 21

  Sector breakdown:
sector
Financials           73
Industrials          73
Consumer Goods       47
Basic Materials      37
Utilities            19
Banks                18
Consumer Services    16
Technology            8
Health Care           7
Oil & Gas             3
✓ Fundamental data loaded


In [34]:
# ── CELL 8: SHARED HELPER — WITHIN-SECTOR TERTILE SORT ───────────────

def assign_within_sector_tertiles(
    df,
    signal_col,
    sector_col,
    ticker_col       = "ticker",
    min_sector_size  = MIN_SECTOR_SIZE,
    higher_is_better = True,
):
    """
    Assigns each stock to P1, P2, or P3 based on within-sector rank.

    For each sector:
      - Top    floor(n/3) stocks → P1  (best signal)
      - Bottom floor(n/3) stocks → P3  (worst signal)
      - Remaining middle stocks  → P2  (neutral)
      - Sectors < min_sector_size → all stocks go to P2

    Parameters:
      df               : DataFrame with ticker, signal, sector columns
      signal_col       : column name of the signal to rank by
      sector_col       : column name of the sector
      min_sector_size  : minimum stocks in sector to perform split
      higher_is_better : True  → high signal = P1 (value: high 1/PE = cheap)
                         True  → high signal = P1 (momentum: high return = strong)

    Returns:
      df with added columns: portfolio, within_sector_rank, within_sector_size
    """
    result = df.copy()
    result["portfolio"]          = "P2"
    result["within_sector_rank"] = np.nan
    result["within_sector_size"] = np.nan

    for sector, group in result.groupby(sector_col):

        valid_idx = group[signal_col].dropna().index
        n         = len(valid_idx)

        result.loc[group.index, "within_sector_size"] = n

        if n < min_sector_size:
            # Too few stocks — assign all to P2 (neutral)
            continue

        # Rank within sector: rank 1 = best signal
        ranked = result.loc[valid_idx, signal_col].rank(
            ascending = not higher_is_better,
            method    = "first"
        )
        result.loc[valid_idx, "within_sector_rank"] = ranked

        # Tertile cutoffs
        n_top = int(np.floor(n / 3))    # P1 size
        n_bot = int(np.floor(n / 3))    # P3 size

        for idx, rank in ranked.items():
            if rank <= n_top:
                result.loc[idx, "portfolio"] = "P1"
            elif rank > n - n_bot:
                result.loc[idx, "portfolio"] = "P3"
            else:
                result.loc[idx, "portfolio"] = "P2"

    return result


def print_portfolio_sizes(df, date_col="date", label=""):
    """Print portfolio size summary."""
    sizes = df.groupby([date_col, "portfolio"]).size().unstack(fill_value=0)
    avg   = sizes.mean()
    print(f"\n  {label} average portfolio sizes:")
    print(f"  P1: {avg.get('P1',0):.0f} stocks  |  "
          f"P2: {avg.get('P2',0):.0f} stocks  |  "
          f"P3: {avg.get('P3',0):.0f} stocks")


print("✓ Helper function loaded")

✓ Helper function loaded


In [35]:
# ── CELL 9: BUILD VALUE PORTFOLIOS (QUARTERLY) ───────────────────────

print("Building value portfolios (quarterly, within-sector sort)...")

def safe_earnings_yield(pe):
    """1/PE for non-financials. Returns NaN for negative or extreme P/E."""
    if pd.isna(pe) or pe <= PE_MIN or pe > PE_MAX:
        return np.nan
    return 1.0 / pe

def safe_book_yield(pb):
    """1/PB for financials. Returns NaN for negative P/B."""
    if pd.isna(pb) or pb <= 0:
        return np.nan
    return 1.0 / pb

all_safe_dates      = sorted(fund["safe_date"].unique())
value_all_quarters  = []

for rebal_date in all_safe_dates:

    # Get most recently published data for each stock
    fund_avail = (
    fund[fund["safe_date"] <= rebal_date]
    .assign(
        value_signal = lambda df: df.apply(
            lambda r: safe_book_yield(r["pb"])
            if r["sector"] in FINANCIAL_SECTORS
            else safe_earnings_yield(r["pe"]),
            axis=1
        ),
        measure = lambda df: df["sector"].apply(
            lambda s: "1/P/B" if s in FINANCIAL_SECTORS else "1/P/E"
        )
    )
    .dropna(subset=["value_signal"])   # ← remove NaN BEFORE picking latest
    .sort_values("safe_date")
    .drop_duplicates(subset="ticker", keep="last")
    .copy()
)

    if len(fund_avail) < 20:
        continue

    # Compute value signal: 1/PB for financials, 1/PE for others
    fund_avail["value_signal"] = fund_avail.apply(
        lambda r: safe_book_yield(r["pb"])
        if r["sector"] in FINANCIAL_SECTORS
        else safe_earnings_yield(r["pe"]),
        axis=1
    )

    fund_avail["measure"] = fund_avail["sector"].apply(
        lambda s: "1/P/B" if s in FINANCIAL_SECTORS else "1/P/E"
    )

    # Within-sector tertile assignment
    quarter_df = assign_within_sector_tertiles(
        df               = fund_avail,
        signal_col       = "value_signal",
        sector_col       = "sector",
        ticker_col       = "ticker",
        higher_is_better = True,  # high 1/PE or 1/PB = cheap = P1
    )

    quarter_df["date"] = rebal_date

    value_all_quarters.append(
        quarter_df[[
            "ticker", "sector", "measure", "value_signal",
            "within_sector_rank", "within_sector_size",
            "portfolio", "date"
        ]]
    )

value_model = pd.concat(value_all_quarters, ignore_index=True)
value_model = value_model.sort_values(
    ["date", "portfolio", "within_sector_rank"]
).reset_index(drop=True)

print(f"\n  Quarters computed: {value_model['date'].nunique()}")
print_portfolio_sizes(value_model, label="Value model")
print("✓ Value portfolios built")

Building value portfolios (quarterly, within-sector sort)...

  Quarters computed: 21

  Value model average portfolio sizes:
  P1: 94 stocks  |  P2: 106 stocks  |  P3: 94 stocks
✓ Value portfolios built


In [36]:
# ── CELL 10: VALUE MODEL — SECTOR COMPOSITION ────────────────────────

latest_q   = value_model["date"].max()
latest_val = value_model[value_model["date"] == latest_q]

print("=" * 70)
print(f"VALUE MODEL — SECTOR COMPOSITION as of {latest_q.date()}")
print("=" * 70)

sectors    = sorted(latest_val["sector"].dropna().unique())

print(f"\n  {'Sector':<25} {'P1':>6} {'P2':>6} {'P3':>6}  {'Total':>6}  Note")
print(f"  {'─'*25} {'─'*6} {'─'*6} {'─'*6}  {'─'*6}  {'─'*15}")

for sector in sectors:
    sec_df = latest_val[latest_val["sector"] == sector]
    counts = sec_df.groupby("portfolio").size()
    p1     = counts.get("P1", 0)
    p2     = counts.get("P2", 0)
    p3     = counts.get("P3", 0)
    total  = p1 + p2 + p3
    note   = "all→P2 (small)" if total < MIN_SECTOR_SIZE else ""
    msure  = "1/P/B" if sector in FINANCIAL_SECTORS else "1/P/E"
    print(f"  {sector:<25} {p1:>6} {p2:>6} {p3:>6}  {total:>6}  {msure} {note}")

totals = latest_val.groupby("portfolio").size()
print(f"  {'─'*25} {'─'*6} {'─'*6} {'─'*6}  {'─'*6}")
print(f"  {'TOTAL':<25} "
      f"{totals.get('P1',0):>6} "
      f"{totals.get('P2',0):>6} "
      f"{totals.get('P3',0):>6}  "
      f"{totals.sum():>6}")

print(f"\n  P2 is slightly larger than P1/P3 because it absorbs")
print(f"  the remainder when sector sizes are not divisible by 3.")

VALUE MODEL — SECTOR COMPOSITION as of 2026-05-15

  Sector                        P1     P2     P3   Total  Note
  ───────────────────────── ────── ────── ──────  ──────  ───────────────
  Banks                          6      6      6      18  1/P/B 
  Basic Materials               12     13     12      37  1/P/E 
  Consumer Goods                15     17     15      47  1/P/E 
  Consumer Services              5      6      5      16  1/P/E 
  Financials                    24     25     24      73  1/P/B 
  Health Care                    2      3      2       7  1/P/E 
  Industrials                   24     25     24      73  1/P/E 
  Oil & Gas                      0      3      0       3  1/P/E all→P2 (small)
  Technology                     2      4      2       8  1/P/E 
  Utilities                      6      7      6      19  1/P/E 
  ───────────────────────── ────── ────── ──────  ──────
  TOTAL                         96    109     96     301

  P2 is slightly larger than P1/P

In [37]:
# Check what quarters exist in fund DataFrame
print("Quarters in fund data:")
print(fund["quarter"].value_counts().sort_index().to_string())

latest_safe_date = fund["safe_date"].max()

latest_quarters = (
    fund.loc[fund["safe_date"] == latest_safe_date, "quarter"]
    .drop_duplicates()
    .tolist()
)

print(f"Latest safe date in fund : {latest_safe_date.date()}")
print(f"Latest quarter in fund   : {latest_quarters}")

Quarters in fund data:
quarter
Q1.2021    301
Q1.2022    301
Q1.2023    301
Q1.2024    301
Q1.2025    301
Q1.2026    301
Q2.2021    301
Q2.2022    301
Q2.2023    301
Q2.2024    301
Q2.2025    301
Q3.2021    301
Q3.2022    301
Q3.2023    301
Q3.2024    301
Q3.2025    301
Q4.2021    301
Q4.2022    301
Q4.2023    301
Q4.2024    301
Q4.2025    301
Latest safe date in fund : 2026-05-15
Latest quarter in fund   : ['Q1.2026']


In [38]:
# ── CELL 11: VALUE MODEL — PREVIEW EACH PORTFOLIO ────────────────────

for pf, label in [
    ("P1", "CHEAP      — top third by value within each sector"),
    ("P2", "MIDDLE     — mid third by value within each sector"),
    ("P3", "EXPENSIVE  — bottom third by value within each sector"),
]:
    pf_df = latest_val[latest_val["portfolio"] == pf].sort_values(
        ["sector", "within_sector_rank"]
    )

    print(f"\n{'─'*70}")
    print(f"  {pf}: {label} | {len(pf_df)} stocks")
    print(f"{'─'*70}")
    print(f"  {'Ticker':<8} {'Sector':<22} {'Measure':<7} "
          f"{'Signal':>8} {'WS Rank':>8} {'WS Size':>8}")
    print(f"  {'─'*8} {'─'*22} {'─'*7} {'─'*8} {'─'*8} {'─'*8}")

    for _, r in pf_df.head(15).iterrows():
        print(
            f"  {r['ticker']:<8} "
            f"{str(r['sector']):<22} "
            f"{r['measure']:<7} "
            f"{r['value_signal']:>8.4f} "
            f"{int(r['within_sector_rank']):>8} "
            f"{int(r['within_sector_size']):>8}"
        )
    if len(pf_df) > 15:
        print(f"  ... and {len(pf_df)-15} more stocks")


──────────────────────────────────────────────────────────────────────
  P1: CHEAP      — top third by value within each sector | 96 stocks
──────────────────────────────────────────────────────────────────────
  Ticker   Sector                 Measure   Signal  WS Rank  WS Size
  ──────── ────────────────────── ─────── ──────── ──────── ────────
  OCB      Banks                  1/P/B     1.1628        1       18
  MSB      Banks                  1/P/B     1.0417        2       18
  TPB      Banks                  1/P/B     1.0000        3       18
  NAB      Banks                  1/P/B     0.9709        4       18
  SHB      Banks                  1/P/B     0.9615        5       18
  VIB      Banks                  1/P/B     0.9009        6       18
  HII      Basic Materials        1/P/E     0.2732        1       37
  TNI      Basic Materials        1/P/E     0.2203        2       37
  VPG      Basic Materials        1/P/E     0.2049        3       37
  AAA      Basic Materials   

In [39]:
# ── CELL 12: VALUE MODEL — TURNOVER AND SAVE ─────────────────────────

# Quarterly turnover by portfolio
print("Value Model — Quarterly Turnover:")
for pf in ["P1", "P2", "P3"]:
    holdings_by_q = (
        value_model[value_model["portfolio"] == pf]
        .groupby("date")["ticker"]
        .apply(set)
    )
    if len(holdings_by_q) > 1:
        dates     = sorted(holdings_by_q.index)
        turnovers = [
            len(holdings_by_q[dates[i-1]].symmetric_difference(
                holdings_by_q[dates[i]]
            )) / max(len(holdings_by_q[dates[i-1]].union(
                holdings_by_q[dates[i]]
            )), 1)
            for i in range(1, len(dates))
        ]
        print(f"  {pf}: mean={np.mean(turnovers):.1%}  "
              f"min={np.min(turnovers):.1%}  "
              f"max={np.max(turnovers):.1%}")

# Save and download
print("\nSaving value model files...")
value_model.to_csv("value_model_3portfolios.csv", index=False)
files.download("value_model_3portfolios.csv")

for pf in ["P1", "P2", "P3"]:
    fname = f"value_model_{pf.lower()}.csv"
    value_model[value_model["portfolio"] == pf].to_csv(fname, index=False)
    files.download(fname)

print("✓ Value model files downloaded")

Value Model — Quarterly Turnover:
  P1: mean=32.7%  min=19.8%  max=50.8%
  P2: mean=45.2%  min=29.7%  max=58.0%
  P3: mean=31.3%  min=15.4%  max=40.7%

Saving value model files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Value model files downloaded


In [60]:
from google.colab import files
import io

print("Upload your market cap file (CSV or Excel)...")
uploaded_mcap = files.upload()

mcap_filename = list(uploaded_mcap.keys())[0]
print(f"\n✓ Uploaded: {mcap_filename}")

Upload your market cap file (CSV or Excel)...


Saving FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (1).xlsx to FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (1) (1).xlsx
Saving FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (2).xlsx to FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (2) (1).xlsx
Saving FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513.xlsx to FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (3).xlsx

✓ Uploaded: FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (1) (1).xlsx


In [61]:
# Column names in your market cap file
MCAP_COL_TICKER = "ticker"
MCAP_COL_DATE   = "Date"
MCAP_COL_MCAP   = "Market cap"

# Date format in your file
# Your sample shows "05/12/2026" — this is ambiguous, choose one:
#   "%d/%m/%Y" → December 5, 2026  (Vietnamese standard)
#   "%m/%d/%Y" → May 12, 2026      (US standard)
MCAP_DATE_FORMAT = "%m/%d/%Y"

print("Configuration:")
print(f"  Ticker column : '{MCAP_COL_TICKER}'")
print(f"  Date column   : '{MCAP_COL_DATE}'")
print(f"  Market cap col: '{MCAP_COL_MCAP}'")
print(f"  Date format   : {MCAP_DATE_FORMAT}")

Configuration:
  Ticker column : 'ticker'
  Date column   : 'Date'
  Market cap col: 'Market cap'
  Date format   : %m/%d/%Y


In [62]:
# ── CELL 3 (FIXED): PARSE ALL 3 MARKET CAP FILES ─────────────────────

import pandas as pd
import numpy as np
import io

print("Parsing all uploaded market cap files...")
print("=" * 60)

all_mcap_frames = []

for fname, content in uploaded_mcap.items():
    # Read CSV or Excel automatically
    if fname.endswith(".xlsx") or fname.endswith(".xls"):
        df = pd.read_excel(io.BytesIO(content))
    else:
        df = pd.read_csv(io.BytesIO(content))

    print(f"\n  File: {fname}")
    print(f"    Raw shape : {df.shape}")
    print(f"    Columns   : {list(df.columns)}")
    print(f"    Unique tickers: {df[MCAP_COL_TICKER].nunique()}")

    all_mcap_frames.append(df)

# Combine all 3 files
mcap_raw = pd.concat(all_mcap_frames, ignore_index=True)

# Rename and clean
mcap_raw = mcap_raw.rename(columns={
    MCAP_COL_TICKER : "ticker",
    MCAP_COL_DATE   : "date",
    MCAP_COL_MCAP   : "mcap",
})

mcap_raw["ticker"] = mcap_raw["ticker"].astype(str).str.strip().str.upper()
mcap_raw["date"]   = pd.to_datetime(mcap_raw["date"], format=MCAP_DATE_FORMAT)
mcap_raw["mcap"]   = pd.to_numeric(mcap_raw["mcap"], errors="coerce")

# Remove duplicates (same ticker + date appearing in multiple files)
mcap_raw = mcap_raw.drop_duplicates(subset=["ticker", "date"], keep="first")

# Drop missing values
mcap_raw = mcap_raw.dropna(subset=["mcap"])
mcap_raw = mcap_raw[mcap_raw["mcap"] > 0]
mcap_raw = mcap_raw.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"\n" + "=" * 60)
print(f"  Combined rows   : {len(mcap_raw):,}")
print(f"  Unique tickers  : {mcap_raw['ticker'].nunique()}")
print(f"  Date range      : {mcap_raw['date'].min().date()} → {mcap_raw['date'].max().date()}")
print("✓ All market cap files combined")

Parsing all uploaded market cap files...

  File: FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (1) (1).xlsx
    Raw shape : (130710, 3)
    Columns   : ['ticker', 'Date', 'Market cap']
    Unique tickers: 100

  File: FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (2) (1).xlsx
    Raw shape : (130031, 3)
    Columns   : ['ticker', 'Date', 'Market cap']
    Unique tickers: 101

  File: FiinProX_DE_Du_lieu_giao_dich_Doanh_nghiep_Theo_ngay_20260513 (3).xlsx
    Raw shape : (129163, 3)
    Columns   : ['ticker', 'Date', 'Market cap']
    Unique tickers: 100

  Combined rows   : 388,717
  Unique tickers  : 301
  Date range      : 2021-02-01 → 2026-05-12
✓ All market cap files combined


In [63]:
"""
Build a wide-format market cap matrix:
  rows = weekly rebalancing dates
  cols = tickers
  values = market cap on that week (forward-filled from latest available)

For each weekly date and each stock, use the MOST RECENT market cap
observation on or before that week.
"""

print("Building weekly market cap matrix...")

# Pivot raw data: rows=date, cols=ticker, values=mcap
mcap_daily = mcap_raw.pivot_table(
    index   = "date",
    columns = "ticker",
    values  = "mcap",
    aggfunc = "last"   # if duplicates, take last
)

# Get weekly dates from price data
week_dates = weekly_prices.index

# Build weekly mcap matrix by forward-filling
# For each weekly date, find most recent mcap on or before that date
mcap_weekly = pd.DataFrame(
    np.nan,
    index   = week_dates,
    columns = mcap_daily.columns
)

# Reindex mcap_daily to include all weekly dates, then forward-fill
all_dates    = sorted(set(mcap_daily.index) | set(week_dates))
mcap_aligned = mcap_daily.reindex(all_dates).ffill()
mcap_weekly  = mcap_aligned.loc[week_dates]

# Coverage check
latest_coverage = mcap_weekly.iloc[-1].notna().sum()
first_valid     = mcap_weekly.notna().any(axis=1).idxmax()
total_cells     = mcap_weekly.shape[0] * mcap_weekly.shape[1]
missing_pct     = mcap_weekly.isna().mean().mean()

print(f"  Weekly mcap matrix shape  : {mcap_weekly.shape}")
print(f"  First week with data      : {first_valid.date()}")
print(f"  Latest week coverage      : {latest_coverage} stocks")
print(f"  Missing cells             : {missing_pct:.1%}")
print("✓ Weekly market cap matrix built")

Building weekly market cap matrix...
  Weekly mcap matrix shape  : (556, 301)
  First week with data      : 2021-02-04
  Latest week coverage      : 301 stocks
  Missing cells             : 50.9%
✓ Weekly market cap matrix built


In [65]:
# ── CELL: VALUE MODEL — QUARTERLY REBALANCING WITH MARKET CAP WEIGHTING ─────

"""
Quarterly value-model portfolio construction.

This version uses the already-built value_model DataFrame:

    value_model columns:
      date
      ticker
      sector
      measure
      value_signal
      portfolio
      within_sector_rank
      within_sector_size

Main logic:
  1. Rebalance only on value_model["date"]
  2. Use latest available market cap on or before each rebalance date
  3. Build market-cap-weighted portfolios
  4. Compute quarterly turnover, fee drag, gross return, and net return
  5. No momentum signal
  6. No combo score
  7. No weekly rebalancing
"""

print("=" * 80)
print("VALUE MODEL — QUARTERLY REBALANCING WITH MARKET CAP WEIGHTS")
print("=" * 80)

# Choose which portfolios to build
VALUE_PORTFOLIOS = ["P1", "P2", "P3"]

# Copy and clean value model
value_model_q = value_model.copy()
value_model_q["date"] = pd.to_datetime(value_model_q["date"])
value_model_q["ticker"] = value_model_q["ticker"].astype(str).str.strip().str.upper()

# Quarterly rebalance dates come directly from value_model
quarter_dates = pd.DatetimeIndex(
    sorted(value_model_q["date"].dropna().unique())
)

print(f"Quarterly rebalance dates: {len(quarter_dates)}")
print(f"Date range: {quarter_dates.min().date()} → {quarter_dates.max().date()}")

# ── Build market-cap matrix ──────────────────────────────────────────

mcap_clean = mcap_raw.copy()
mcap_clean["date"] = pd.to_datetime(mcap_clean["date"])
mcap_clean["ticker"] = mcap_clean["ticker"].astype(str).str.strip().str.upper()
mcap_clean["mcap"] = pd.to_numeric(mcap_clean["mcap"], errors="coerce")

mcap_clean = (
    mcap_clean
    .dropna(subset=["date", "ticker", "mcap"])
    .query("mcap > 0")
    .sort_values(["ticker", "date"])
)

mcap_wide = (
    mcap_clean
    .pivot_table(
        index="date",
        columns="ticker",
        values="mcap",
        aggfunc="last"
    )
    .sort_index()
)

# Latest available market cap on or before each quarterly rebalance date
mcap_at_rebal = mcap_wide.reindex(quarter_dates, method="ffill")


# ── Build price matrix at quarterly rebalance dates ──────────────────

daily_wide_q = daily_wide.copy()
daily_wide_q.index = pd.to_datetime(daily_wide_q.index)
daily_wide_q = daily_wide_q.sort_index()

# Latest available price on or before each rebalance date
price_at_rebal = daily_wide_q.reindex(quarter_dates, method="ffill")


def get_period_returns(start_date, end_date):
    """
    Return from start_date to end_date using prices available
    on or before each date.
    """
    start_px = price_at_rebal.loc[start_date]
    end_px   = price_at_rebal.loc[end_date]

    ret = end_px / start_px - 1
    ret = ret.replace([np.inf, -np.inf], np.nan)

    return ret


def compute_sector_replicated_mcap_weights(assigned_df, portfolio_name):
    """
    Compute market-cap weights for one value portfolio.

    Weighting method:
      1. Compute sector weights from the full eligible universe.
      2. Keep only sectors represented in the target portfolio.
      3. Redistribute unavailable sector weights across represented sectors.
      4. Within each represented sector, weight stocks by market cap.

    This preserves your previous combined-model idea of sector-replicated
    market-cap weighting.
    """

    eligible = assigned_df.copy()

    eligible = eligible.dropna(subset=["ticker", "sector", "mcap"])
    eligible = eligible[eligible["mcap"] > 0]

    pf_df = eligible[eligible["portfolio"] == portfolio_name].copy()

    if pf_df.empty:
        return pf_df

    # Sector weights from full eligible universe
    universe_sector_mcap = eligible.groupby("sector")["mcap"].sum()
    universe_total_mcap  = universe_sector_mcap.sum()

    if universe_total_mcap <= 0:
        return pd.DataFrame()

    sector_weight_universe = universe_sector_mcap / universe_total_mcap

    # Market cap inside this portfolio by sector
    pf_sector_mcap = pf_df.groupby("sector")["mcap"].sum()

    represented_sectors = pf_sector_mcap[pf_sector_mcap > 0].index

    sector_weight_target = sector_weight_universe.reindex(
        represented_sectors
    ).fillna(0)

    if sector_weight_target.sum() <= 0:
        return pd.DataFrame()

    # Redistribute weights only across sectors represented in this portfolio
    sector_weight_target = sector_weight_target / sector_weight_target.sum()

    pf_df["sector_weight_target"] = pf_df["sector"].map(sector_weight_target)
    pf_df["portfolio_sector_mcap"] = pf_df["sector"].map(pf_sector_mcap)

    pf_df["weight_within_sector"] = (
        pf_df["mcap"] / pf_df["portfolio_sector_mcap"]
    )

    pf_df["weight"] = (
        pf_df["sector_weight_target"] *
        pf_df["weight_within_sector"]
    )

    pf_df["weight"] = (
        pf_df["weight"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    if pf_df["weight"].sum() > 0:
        pf_df["weight"] = pf_df["weight"] / pf_df["weight"].sum()
    else:
        return pd.DataFrame()

    return pf_df


# ── Quarterly rebalance loop ─────────────────────────────────────────

all_quarter_rows = []
all_stock_rows   = []

prev_weights_by_pf = {pf: {} for pf in VALUE_PORTFOLIOS}
prev_date_by_pf    = {pf: None for pf in VALUE_PORTFOLIOS}

for i, date in enumerate(quarter_dates):

    # Current value-model assignments
    quarter_df = value_model_q[value_model_q["date"] == date].copy()

    if quarter_df.empty:
        continue

    # Attach market cap available at construction date
    if date not in mcap_at_rebal.index:
        continue

    mcap_row = mcap_at_rebal.loc[date]

    quarter_df["mcap"] = quarter_df["ticker"].apply(
        lambda t: mcap_row.get(t, np.nan)
    )

    quarter_df = quarter_df.dropna(subset=["mcap"])
    quarter_df = quarter_df[quarter_df["mcap"] > 0].copy()

    if len(quarter_df) < 20:
        continue

    # Next quarter return period
    if i < len(quarter_dates) - 1:
        next_date = quarter_dates[i + 1]
        ret_current_to_next = get_period_returns(date, next_date)
    else:
        next_date = pd.NaT
        ret_current_to_next = None

    for pf in VALUE_PORTFOLIOS:

        pf_df = compute_sector_replicated_mcap_weights(
            assigned_df=quarter_df,
            portfolio_name=pf
        )

        if pf_df.empty:
            continue

        curr_weights = dict(zip(pf_df["ticker"], pf_df["weight"]))
        curr_tickers = set(curr_weights.keys())

        prev_weights = prev_weights_by_pf.get(pf, {})
        prev_tickers = set(prev_weights.keys())
        prev_date    = prev_date_by_pf.get(pf)

        total_mcap = pf_df["mcap"].sum()

        # ── Fee and turnover calculation ─────────────────────────────

        if prev_date is not None and len(prev_weights) > 0:

            entering = curr_tickers - prev_tickers
            exiting  = prev_tickers - curr_tickers
            holding  = curr_tickers & prev_tickers

            # Drift previous weights from previous rebalance date to current date
            ret_prev_to_current = get_period_returns(prev_date, date)

            drifted_values = {}

            for t, prev_w in prev_weights.items():
                r = ret_prev_to_current.get(t, 0.0)

                if pd.isna(r):
                    r = 0.0

                drifted_values[t] = prev_w * (1.0 + r)

            total_drifted_value = sum(drifted_values.values())

            if total_drifted_value > 0:
                drifted_weights = {
                    t: v / total_drifted_value
                    for t, v in drifted_values.items()
                }
            else:
                drifted_weights = {
                    t: 0.0
                    for t in prev_weights.keys()
                }

            # Buy cost for entering stocks
            buy_cost = sum(
                curr_weights[t] * (BUY_FEE_PCT / 100)
                for t in entering
            )

            # Sell cost for exiting stocks
            sell_cost = sum(
                drifted_weights.get(t, 0.0) *
                ((SELL_FEE_PCT + SELL_TAX_PCT) / 100)
                for t in exiting
            )

            # Rebalance cost for existing holdings
            rebal_cost = 0.0

            for t in holding:
                target_w  = curr_weights.get(t, 0.0)
                drifted_w = drifted_weights.get(t, 0.0)

                active_trade_w = target_w - drifted_w

                if active_trade_w > 0:
                    rebal_cost += active_trade_w * (BUY_FEE_PCT / 100)

                elif active_trade_w < 0:
                    rebal_cost += abs(active_trade_w) * (
                        (SELL_FEE_PCT + SELL_TAX_PCT) / 100
                    )

            fee_drag = buy_cost + sell_cost + rebal_cost

            # Turnover based on active traded weight
            entering_turnover = sum(
                curr_weights[t]
                for t in entering
            )

            exiting_turnover = sum(
                drifted_weights.get(t, 0.0)
                for t in exiting
            )

            holding_turnover = 0.0

            for t in holding:
                target_w  = curr_weights.get(t, 0.0)
                drifted_w = drifted_weights.get(t, 0.0)

                holding_turnover += abs(target_w - drifted_w)

            turnover = (
                entering_turnover +
                exiting_turnover +
                holding_turnover
            ) / 2

        else:
            # First rebalance: buy the whole portfolio
            entering = curr_tickers
            exiting  = set()

            buy_cost = sum(
                curr_weights[t] * (BUY_FEE_PCT / 100)
                for t in curr_tickers
            )

            sell_cost  = 0.0
            rebal_cost = 0.0
            fee_drag   = buy_cost
            turnover   = 1.0

        # ── Quarterly return from current rebalance date to next date ─

        if ret_current_to_next is not None:
            weighted_ret = 0.0
            used_weight  = 0.0

            for ticker, weight in curr_weights.items():
                r = ret_current_to_next.get(ticker, np.nan)

                if not pd.isna(r):
                    weighted_ret += weight * r
                    used_weight  += weight

            gross_ret = weighted_ret / used_weight if used_weight > 0 else np.nan
            net_ret   = gross_ret - fee_drag if not pd.isna(gross_ret) else np.nan

        else:
            gross_ret = np.nan
            net_ret   = np.nan

        # ── Save quarterly portfolio summary ─────────────────────────

        all_quarter_rows.append({
            "date"       : date,
            "period_end" : next_date,
            "portfolio"  : pf,
            "n_stocks"   : len(curr_tickers),
            "n_entering" : len(entering),
            "n_exiting"  : len(exiting),
            "turnover"   : turnover,
            "buy_cost"   : buy_cost,
            "sell_cost"  : sell_cost,
            "rebal_cost" : rebal_cost,
            "fee_drag"   : fee_drag,
            "gross_ret"  : gross_ret,
            "net_ret"    : net_ret,
            "total_mcap" : total_mcap,
        })

        # ── Save stock-level weights ─────────────────────────────────

        pf_df["date"]       = date
        pf_df["period_end"] = next_date

        pf_df["status"] = pf_df["ticker"].apply(
            lambda t: "NEW" if t in entering else "HOLD"
        )

        all_stock_rows.append(
            pf_df[[
                "date", "period_end",
                "ticker", "sector", "measure",
                "portfolio", "value_signal",
                "within_sector_rank", "within_sector_size",
                "mcap", "sector_weight_target",
                "weight_within_sector", "weight",
                "status"
            ]]
        )

        # Update previous weights for this portfolio
        prev_weights_by_pf[pf] = curr_weights
        prev_date_by_pf[pf]    = date


# ── Build final output DataFrames ────────────────────────────────────

value_quarterly_results = pd.DataFrame(all_quarter_rows)

value_quarterly_stock_assign = pd.concat(
    all_stock_rows,
    ignore_index=True
)

value_quarterly_stock_assign = value_quarterly_stock_assign.sort_values(
    ["date", "portfolio", "weight"],
    ascending=[True, True, False]
).reset_index(drop=True)

print("\n✓ Quarterly value-model market-cap-weighted portfolios built")

print(f"\nQuarters computed: {value_quarterly_results['date'].nunique()}")

print("\nAverage portfolio size:")
print(
    value_quarterly_results
    .groupby("portfolio")["n_stocks"]
    .mean()
    .round(1)
    .to_string()
)

print("\nAverage quarterly turnover:")
print(
    value_quarterly_results
    .groupby("portfolio")["turnover"]
    .mean()
    .map(lambda x: f"{x:.1%}")
    .to_string()
)

print("\nAnnualized fee drag:")
print(
    value_quarterly_results
    .groupby("portfolio")["fee_drag"]
    .mean()
    .mul(4)
    .map(lambda x: f"{x:.2%}")
    .to_string()
)

print("\nAnnualized gross return:")
print(
    value_quarterly_results
    .groupby("portfolio")["gross_ret"]
    .mean()
    .mul(4)
    .map(lambda x: f"{x:.2%}")
    .to_string()
)

print("\nAnnualized net return:")
print(
    value_quarterly_results
    .groupby("portfolio")["net_ret"]
    .mean()
    .mul(4)
    .map(lambda x: f"{x:.2%}")
    .to_string()
)

VALUE MODEL — QUARTERLY REBALANCING WITH MARKET CAP WEIGHTS
Quarterly rebalance dates: 21
Date range: 2021-05-15 → 2026-05-15

✓ Quarterly value-model market-cap-weighted portfolios built

Quarters computed: 21

Average portfolio size:
portfolio
P1     93.8
P2    106.0
P3     94.0

Average quarterly turnover:
portfolio
P1    35.0%
P2    35.6%
P3    20.4%

Annualized fee drag:
portfolio
P1    0.64%
P2    0.66%
P3    0.35%

Annualized gross return:
portfolio
P1     2.83%
P2    10.16%
P3    18.38%

Annualized net return:
portfolio
P1     2.18%
P2     9.49%
P3    18.01%


In [67]:
# ── CELL: VALUE MODEL — LATEST QUARTER PORTFOLIO WITH WEIGHTS ────────

latest_q = value_quarterly_stock_assign["date"].max()
latest_value_q = value_quarterly_stock_assign[
    value_quarterly_stock_assign["date"] == latest_q
]

def fmt_rank(x):
    """Safely format within-sector rank."""
    if pd.isna(x):
        return "-"
    return str(int(x))

def fmt_float(x, decimals=4):
    """Safely format float values."""
    if pd.isna(x):
        return "-"
    return f"{x:.{decimals}f}"

print("=" * 85)
print(f"VALUE MODEL — MARKET-CAP-WEIGHTED PORTFOLIOS as of {latest_q.date()}")
print("=" * 85)

for pf, label in [
    ("P1", "CHEAP"),
    ("P2", "MIDDLE"),
    ("P3", "EXPENSIVE"),
]:
    pf_df = latest_value_q[
        latest_value_q["portfolio"] == pf
    ].sort_values("weight", ascending=False)

    print(f"\n{'─' * 85}")
    print(f"{pf}: {label} | {len(pf_df)} stocks")
    print(f"{'─' * 85}")

    print(
        f"{'Ticker':<8} "
        f"{'Sector':<24} "
        f"{'Measure':<8} "
        f"{'Weight':>8} "
        f"{'Mcap (B)':>12} "
        f"{'Value Sig':>10} "
        f"{'Rank':>6} "
        f"{'Status':<6}"
    )

    print(
        f"{'─'*8} "
        f"{'─'*24} "
        f"{'─'*8} "
        f"{'─'*8} "
        f"{'─'*12} "
        f"{'─'*10} "
        f"{'─'*6} "
        f"{'─'*6}"
    )

    for _, r in pf_df.head(20).iterrows():
        print(
            f"{r['ticker']:<8} "
            f"{str(r['sector']):<24} "
            f"{str(r['measure']):<8} "
            f"{r['weight']:>7.2%} "
            f"{r['mcap']/1e9:>11.0f} "
            f"{fmt_float(r['value_signal'], 4):>10} "
            f"{fmt_rank(r['within_sector_rank']):>6} "
            f"{str(r['status']):<6}"
        )

    if len(pf_df) > 20:
        print(f"... and {len(pf_df) - 20} more stocks")

    print(f"\nTotal weight      : {pf_df['weight'].sum():.2%}")
    print(f"Total market cap  : {pf_df['mcap'].sum()/1e12:.2f} trillion VND")
    print(f"Top 5 weight      : {pf_df['weight'].head(5).sum():.1%}")
    print(f"Top 10 weight     : {pf_df['weight'].head(10).sum():.1%}")

VALUE MODEL — MARKET-CAP-WEIGHTED PORTFOLIOS as of 2026-05-15

─────────────────────────────────────────────────────────────────────────────────────
P1: CHEAP | 96 stocks
─────────────────────────────────────────────────────────────────────────────────────
Ticker   Sector                   Measure    Weight     Mcap (B)  Value Sig   Rank Status
──────── ──────────────────────── ──────── ──────── ──────────── ────────── ────── ──────
SHB      Banks                    1/P/B      9.09%       73543     0.9615      5 HOLD  
VIB      Banks                    1/P/B      6.73%       54464     0.9009      6 NEW   
DXS      Financials               1/P/B      5.88%        4749     1.4706     21 HOLD  
TPB      Banks                    1/P/B      5.42%       43830     1.0000      3 HOLD  
MSB      Banks                    1/P/B      5.17%       41808     1.0417      2 HOLD  
CRE      Financials               1/P/B      4.36%        3524     1.6393     17 HOLD  
OCB      Banks                    1

In [68]:
# ── CELL: SAVE VALUE MODEL QUARTERLY MARKET-CAP-WEIGHTED OUTPUTS ─────

print("Saving quarterly value-model market-cap-weighted files...")

value_quarterly_results.to_csv(
    "value_model_quarterly_market_cap_weighted_results.csv",
    index=False
)

value_quarterly_stock_assign.to_csv(
    "value_model_quarterly_market_cap_weighted_stock_assignments.csv",
    index=False
)

files.download("value_model_quarterly_market_cap_weighted_results.csv")
files.download("value_model_quarterly_market_cap_weighted_stock_assignments.csv")

for pf in ["P1", "P2", "P3"]:
    fname = f"value_model_quarterly_market_cap_weighted_{pf.lower()}.csv"

    value_quarterly_stock_assign[
        value_quarterly_stock_assign["portfolio"] == pf
    ].to_csv(fname, index=False)

    files.download(fname)

print("✓ Quarterly value-model market-cap-weighted files downloaded")

Saving quarterly value-model market-cap-weighted files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Quarterly value-model market-cap-weighted files downloaded


In [40]:
# ── CELL 13: BUILD WEEKLY MOMENTUM SIGNAL ────────────────────────────

print("Computing weekly momentum signal...")

def compute_weekly_momentum(price_df, start_lag, end_lag):
    """
    Cumulative return from (t - end_lag) to (t - start_lag) weeks.
    skip last start_lag weeks to avoid short-term reversal.
    """
    result = pd.DataFrame(
        index   = price_df.index,
        columns = price_df.columns,
        dtype   = float
    )
    for t in range(end_lag, len(price_df)):
        p_start        = price_df.iloc[t - end_lag]
        p_end          = price_df.iloc[t - start_lag]
        result.iloc[t] = (p_end / p_start) - 1
    return result

weekly_mom = compute_weekly_momentum(
    weekly_prices, MOM_WEEK_START, MOM_WEEK_END
)

# Live sector assignment per ticker (re-read each period)
sector_live = (
    fund.sort_values("safe_date")
    .drop_duplicates("ticker", keep="last")
    .set_index("ticker")["sector"]
)

print(f"  Weekly price matrix : {weekly_prices.shape}")
print(f"  Latest week stocks  : {weekly_mom.iloc[-1].notna().sum()}")
print("✓ Weekly momentum built")

Computing weekly momentum signal...
  Weekly price matrix : (556, 301)
  Latest week stocks  : 298
✓ Weekly momentum built


In [45]:
# ── CELL 14 (FIXED): WEEKLY REBALANCING — 3 PORTFOLIOS ───────────────

print("Running weekly rebalancing...")

week_dates      = weekly_mom.index
all_week_rows   = []      # portfolio-level summary (fees, returns)
all_stock_rows  = []      # stock-level assignments (ticker per portfolio)

for i, date in enumerate(week_dates):

    mom_row = weekly_mom.loc[date].dropna()
    if len(mom_row) < 20:
        continue

    # Build weekly universe DataFrame
    week_df = pd.DataFrame({
        "ticker"     : mom_row.index,
        "mom_signal" : mom_row.values,
        "sector"     : [sector_live.get(t, "Unknown") for t in mom_row.index],
    })

    # Within-sector tertile assignment
    assigned = assign_within_sector_tertiles(
        df               = week_df,
        signal_col       = "mom_signal",
        sector_col       = "sector",
        ticker_col       = "ticker",
        higher_is_better = True,
    )

    # ── Save individual stock assignments ─────────────────────────────
    assigned["date"] = date
    all_stock_rows.append(
        assigned[[
            "ticker", "sector", "mom_signal",
            "within_sector_rank", "within_sector_size",
            "portfolio", "date"
        ]]
    )

    # ── Compute returns and fees ──────────────────────────────────────
    if i > 0 and date in weekly_returns.index:
        week_ret = weekly_returns.loc[date]

        if i > 0:
            prev_date = week_dates[i - 1]
            prev_row  = weekly_mom.loc[prev_date].dropna()
            if len(prev_row) >= 20:
                prev_df = pd.DataFrame({
                    "ticker"     : prev_row.index,
                    "mom_signal" : prev_row.values,
                    "sector"     : [sector_live.get(t, "Unknown")
                                   for t in prev_row.index],
                })
                prev_assigned = assign_within_sector_tertiles(
                    df               = prev_df,
                    signal_col       = "mom_signal",
                    sector_col       = "sector",
                    ticker_col       = "ticker",
                    higher_is_better = True,
                ).set_index("ticker")["portfolio"]

                curr_assigned = assigned.set_index("ticker")["portfolio"]
                common        = prev_assigned.index.intersection(curr_assigned.index)
                n_changed     = (prev_assigned[common] != curr_assigned[common]).sum()
                n_total       = max(len(curr_assigned), 1)
                turnover      = n_changed / n_total
                fee_drag      = (
                    (n_changed / n_total) *
                    (BUY_FEE_PCT + SELL_FEE_PCT + SELL_TAX_PCT) / 100
                )
            else:
                turnover = fee_drag = 0.0
        else:
            turnover = fee_drag = 0.0

        for pf in ["P1", "P2", "P3"]:
            pf_tickers = assigned[assigned["portfolio"] == pf]["ticker"].tolist()
            pf_rets    = [
                week_ret.get(t, np.nan) for t in pf_tickers
                if t in week_ret.index and not np.isnan(week_ret[t])
            ]
            gross_ret = np.mean(pf_rets) if pf_rets else 0.0
            net_ret   = gross_ret - fee_drag

            all_week_rows.append({
                "date"      : date,
                "portfolio" : pf,
                "n_stocks"  : len(pf_tickers),
                "turnover"  : turnover,
                "fee_drag"  : fee_drag,
                "gross_ret" : gross_ret,
                "net_ret"   : net_ret,
            })

# Build both DataFrames
mom_results      = pd.DataFrame(all_week_rows)
mom_stock_assign = pd.concat(all_stock_rows, ignore_index=True)
mom_stock_assign = mom_stock_assign.sort_values(
    ["date", "portfolio", "within_sector_rank"]
).reset_index(drop=True)

print(f"\n  Weeks computed: {mom_results['date'].nunique()}")
print(f"\n  {'Portfolio':<12} {'Avg Size':>9} {'Avg Turn':>10} "
      f"{'Ann Fee':>9} {'Ann Gross':>11} {'Ann Net':>10}")
print(f"  {'─'*12} {'─'*9} {'─'*10} {'─'*9} {'─'*11} {'─'*10}")

for pf in ["P1", "P2", "P3"]:
    pf_df = mom_results[mom_results["portfolio"] == pf]
    print(
        f"  {pf:<12} "
        f"{pf_df['n_stocks'].mean():>9.0f} "
        f"{pf_df['turnover'].mean():>9.1%} "
        f"{pf_df['fee_drag'].mean()*52:>8.2%} "
        f"{pf_df['gross_ret'].mean()*52:>10.2%} "
        f"{pf_df['net_ret'].mean()*52:>9.2%}"
    )
print("✓ Momentum portfolios built")

Running weekly rebalancing...

  Weeks computed: 553

  Portfolio     Avg Size   Avg Turn   Ann Fee   Ann Gross    Ann Net
  ──────────── ───────── ────────── ───────── ─────────── ──────────
  P1                  86     64.3%   16.72%    273.81%   257.09%
  P2                  98     64.3%   16.72%     -2.08%   -18.80%
  P3                  86     64.3%   16.72%   -214.42%  -231.14%
✓ Momentum portfolios built


In [47]:
# ── CELL 15: MOMENTUM MODEL — LATEST WEEK HOLDINGS ───────────────────

latest_week = weekly_mom.index[-1]
mom_latest  = weekly_mom.loc[latest_week].dropna()

latest_week_df = pd.DataFrame({
    "ticker"     : mom_latest.index,
    "mom_signal" : mom_latest.values,
    "sector"     : [sector_live.get(t, "Unknown") for t in mom_latest.index],
})

latest_assigned = assign_within_sector_tertiles(
    df               = latest_week_df,
    signal_col       = "mom_signal",
    sector_col       = "sector",
    ticker_col       = "ticker",
    higher_is_better = True,
)

print("=" * 70)
print(f"MOMENTUM MODEL — HOLDINGS as of {latest_week.date()}")
print("=" * 70)

for pf, label in [
    ("P1", "STRONG MOMENTUM — top third within each sector"),
    ("P2", "MIDDLE         — mid third within each sector"),
    ("P3", "WEAK MOMENTUM  — bottom third within each sector"),
]:
    pf_df = latest_assigned[latest_assigned["portfolio"] == pf].sort_values(
        ["sector", "within_sector_rank"]
    )

    print(f"\n  {pf}: {label} | {len(pf_df)} stocks")
    print(f"  {'Ticker':<8} {'Sector':<22} {'2W Return':>12} {'WS Rank':>9}")
    print(f"  {'─'*8} {'─'*22} {'─'*12} {'─'*9}")

    for _, r in pf_df.head(15).iterrows():
        print(
            f"  {r['ticker']:<8} "
            f"{str(r['sector']):<22} "
            f"{r['mom_signal']:>11.2%} "
            f"{int(r['within_sector_rank']):>9}"
        )
    if len(pf_df) > 15:
        print(f"  ... and {len(pf_df)-15} more stocks")

MOMENTUM MODEL — HOLDINGS as of 2026-05-21

  P1: STRONG MOMENTUM — top third within each sector | 95 stocks
  Ticker   Sector                    2W Return   WS Rank
  ──────── ────────────────────── ──────────── ─────────
  BID      Banks                        4.02%         1
  VCB      Banks                        3.61%         2
  CTG      Banks                        0.97%         3
  ACB      Banks                        0.88%         4
  LPB      Banks                        0.38%         5
  MSB      Banks                        0.36%         6
  GVR      Basic Materials              8.56%         1
  PHR      Basic Materials              5.40%         2
  DPM      Basic Materials              4.65%         3
  DCM      Basic Materials              4.53%         4
  PTB      Basic Materials              2.85%         5
  TRC      Basic Materials              1.99%         6
  BFC      Basic Materials              1.94%         7
  NHH      Basic Materials              1.93%    

In [69]:
# ── CELL 14: MOMENTUM MODEL — WEEKLY REBALANCING WITH MARKET CAP WEIGHTING ──

print("=" * 80)
print("MOMENTUM MODEL — WEEKLY REBALANCING WITH MARKET CAP WEIGHTS")
print("=" * 80)

MOM_PORTFOLIOS = ["P1", "P2", "P3"]

# ── Safety checks ────────────────────────────────────────────────────

required_objects = [
    "weekly_mom",
    "weekly_prices",
    "mcap_raw",
    "sector_live",
    "assign_within_sector_tertiles",
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"Missing required object: {obj}")

# ── Prepare weekly rebalance dates ───────────────────────────────────

week_dates = pd.DatetimeIndex(weekly_mom.index).sort_values()

# ── Prepare market cap matrix ────────────────────────────────────────

mcap_clean = mcap_raw.copy()

mcap_clean["date"] = pd.to_datetime(mcap_clean["date"])
mcap_clean["ticker"] = (
    mcap_clean["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)
mcap_clean["mcap"] = pd.to_numeric(mcap_clean["mcap"], errors="coerce")

mcap_clean = (
    mcap_clean
    .dropna(subset=["date", "ticker", "mcap"])
    .query("mcap > 0")
    .sort_values(["ticker", "date"])
)

mcap_wide = (
    mcap_clean
    .pivot_table(
        index="date",
        columns="ticker",
        values="mcap",
        aggfunc="last"
    )
    .sort_index()
)

# Track actual market-cap observation date
mcap_date_marker = pd.DataFrame(
    np.tile(mcap_wide.index.values.reshape(-1, 1), (1, len(mcap_wide.columns))),
    index=mcap_wide.index,
    columns=mcap_wide.columns
)

mcap_date_marker = mcap_date_marker.where(mcap_wide.notna())

# Forward-fill market cap to weekly rebalance dates
all_mcap_dates = mcap_wide.index.union(week_dates).sort_values()

mcap_weekly = (
    mcap_wide
    .reindex(all_mcap_dates)
    .ffill()
    .reindex(week_dates)
)

mcap_date_weekly = (
    mcap_date_marker
    .reindex(all_mcap_dates)
    .ffill()
    .reindex(week_dates)
)

# ── Prepare weekly price matrix for holding-period returns ───────────

weekly_prices_mom = weekly_prices.copy()
weekly_prices_mom.index = pd.to_datetime(weekly_prices_mom.index)
weekly_prices_mom = weekly_prices_mom.sort_index()

def get_week_returns(start_date, end_date):
    """
    Return from start_date to end_date using weekly prices.
    Used for:
      1. portfolio holding-period return
      2. drifted weights before rebalancing
    """
    if start_date not in weekly_prices_mom.index:
        return pd.Series(dtype=float)

    if end_date not in weekly_prices_mom.index:
        return pd.Series(dtype=float)

    start_px = weekly_prices_mom.loc[start_date]
    end_px   = weekly_prices_mom.loc[end_date]

    ret = end_px / start_px - 1
    ret = ret.replace([np.inf, -np.inf], np.nan)

    return ret


def compute_sector_replicated_mcap_weights(assigned_df, portfolio_name):
    """
    Compute market-cap weights for one momentum portfolio.

    Weighting method:
      1. Compute sector weights from the full eligible universe.
      2. Keep only sectors represented inside the target portfolio.
      3. Redistribute unavailable sector weights across represented sectors.
      4. Within each sector, weight stocks by market cap.

    Final stock weight:
      weight_i =
          sector_weight_target
          × mcap_i / total_mcap_of_portfolio_sector
    """

    eligible = assigned_df.copy()

    eligible = eligible.dropna(subset=["ticker", "sector", "mcap"])
    eligible = eligible[eligible["mcap"] > 0]

    pf_df = eligible[eligible["portfolio"] == portfolio_name].copy()

    if pf_df.empty:
        return pd.DataFrame()

    # Sector weights from full eligible universe
    universe_sector_mcap = eligible.groupby("sector")["mcap"].sum()
    universe_total_mcap  = universe_sector_mcap.sum()

    if universe_total_mcap <= 0:
        return pd.DataFrame()

    sector_weight_universe = universe_sector_mcap / universe_total_mcap

    # Market cap inside this portfolio by sector
    pf_sector_mcap = pf_df.groupby("sector")["mcap"].sum()

    represented_sectors = pf_sector_mcap[pf_sector_mcap > 0].index

    sector_weight_target = sector_weight_universe.reindex(
        represented_sectors
    ).fillna(0)

    if sector_weight_target.sum() <= 0:
        return pd.DataFrame()

    # Redistribute only across sectors represented in this portfolio
    sector_weight_target = sector_weight_target / sector_weight_target.sum()

    pf_df["sector_weight_target"] = pf_df["sector"].map(sector_weight_target)
    pf_df["portfolio_sector_mcap"] = pf_df["sector"].map(pf_sector_mcap)

    pf_df["weight_within_sector"] = (
        pf_df["mcap"] / pf_df["portfolio_sector_mcap"]
    )

    pf_df["weight"] = (
        pf_df["sector_weight_target"] *
        pf_df["weight_within_sector"]
    )

    pf_df["weight"] = (
        pf_df["weight"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    if pf_df["weight"].sum() > 0:
        pf_df["weight"] = pf_df["weight"] / pf_df["weight"].sum()
    else:
        return pd.DataFrame()

    return pf_df


# ── Weekly rebalance loop ────────────────────────────────────────────

all_week_rows  = []
all_stock_rows = []

prev_weights_by_pf = {pf: {} for pf in MOM_PORTFOLIOS}
prev_date_by_pf    = {pf: None for pf in MOM_PORTFOLIOS}

print("Running weekly momentum rebalancing with market cap weights...")

for i, date in enumerate(week_dates):

    mom_row = weekly_mom.loc[date].dropna()

    if len(mom_row) < 20:
        continue

    if date not in mcap_weekly.index:
        continue

    mcap_row      = mcap_weekly.loc[date]
    mcap_date_row = mcap_date_weekly.loc[date]

    # Build weekly universe
    week_df = pd.DataFrame({
        "ticker"     : mom_row.index,
        "mom_signal" : mom_row.values,
        "sector"     : [sector_live.get(t, "Unknown") for t in mom_row.index],
        "mcap"       : [mcap_row.get(t, np.nan) for t in mom_row.index],
        "mcap_date"  : [mcap_date_row.get(t, pd.NaT) for t in mom_row.index],
    })

    week_df = week_df.dropna(subset=["mom_signal", "mcap"])
    week_df = week_df[week_df["mcap"] > 0].copy()

    if len(week_df) < 20:
        continue

    # Assign P1/P2/P3 by momentum signal only
    assigned = assign_within_sector_tertiles(
        df               = week_df,
        signal_col       = "mom_signal",
        sector_col       = "sector",
        ticker_col       = "ticker",
        min_sector_size  = MIN_SECTOR_SIZE,
        higher_is_better = True,
    )

    # Holding-period return from this week to next week
    if i < len(week_dates) - 1:
        next_date = week_dates[i + 1]
        ret_current_to_next = get_week_returns(date, next_date)
    else:
        next_date = pd.NaT
        ret_current_to_next = None

    for pf in MOM_PORTFOLIOS:

        pf_df = compute_sector_replicated_mcap_weights(
            assigned_df=assigned,
            portfolio_name=pf
        )

        if pf_df.empty:
            continue

        curr_weights = dict(zip(pf_df["ticker"], pf_df["weight"]))
        curr_tickers = set(curr_weights.keys())

        prev_weights = prev_weights_by_pf.get(pf, {})
        prev_tickers = set(prev_weights.keys())
        prev_date    = prev_date_by_pf.get(pf)

        total_mcap = pf_df["mcap"].sum()

        # ── Fee and turnover calculation ─────────────────────────────

        if prev_date is not None and len(prev_weights) > 0:

            entering = curr_tickers - prev_tickers
            exiting  = prev_tickers - curr_tickers
            holding  = curr_tickers & prev_tickers

            # Drift previous weights from previous rebalance date to current date
            ret_prev_to_current = get_week_returns(prev_date, date)

            drifted_values = {}

            for t, prev_w in prev_weights.items():
                r = ret_prev_to_current.get(t, 0.0)

                if pd.isna(r):
                    r = 0.0

                drifted_values[t] = prev_w * (1.0 + r)

            total_drifted_value = sum(drifted_values.values())

            if total_drifted_value > 0:
                drifted_weights = {
                    t: v / total_drifted_value
                    for t, v in drifted_values.items()
                }
            else:
                drifted_weights = {
                    t: 0.0
                    for t in prev_weights.keys()
                }

            # Entering stocks: buy current target weight
            buy_cost = sum(
                curr_weights[t] * (BUY_FEE_PCT / 100)
                for t in entering
            )

            # Exiting stocks: sell drifted pre-rebalance weight
            sell_cost = sum(
                drifted_weights.get(t, 0.0) *
                ((SELL_FEE_PCT + SELL_TAX_PCT) / 100)
                for t in exiting
            )

            # Holding stocks: cost only on active rebalance trade
            rebal_cost = 0.0

            for t in holding:
                target_w  = curr_weights.get(t, 0.0)
                drifted_w = drifted_weights.get(t, 0.0)

                active_trade_w = target_w - drifted_w

                if active_trade_w > 0:
                    rebal_cost += active_trade_w * (BUY_FEE_PCT / 100)

                elif active_trade_w < 0:
                    rebal_cost += abs(active_trade_w) * (
                        (SELL_FEE_PCT + SELL_TAX_PCT) / 100
                    )

            fee_drag = buy_cost + sell_cost + rebal_cost

            # Turnover
            entering_turnover = sum(
                curr_weights[t]
                for t in entering
            )

            exiting_turnover = sum(
                drifted_weights.get(t, 0.0)
                for t in exiting
            )

            holding_turnover = 0.0

            for t in holding:
                target_w  = curr_weights.get(t, 0.0)
                drifted_w = drifted_weights.get(t, 0.0)

                holding_turnover += abs(target_w - drifted_w)

            turnover = (
                entering_turnover +
                exiting_turnover +
                holding_turnover
            ) / 2

        else:
            # First rebalance: buy the full portfolio
            entering = curr_tickers
            exiting  = set()

            buy_cost = sum(
                curr_weights[t] * (BUY_FEE_PCT / 100)
                for t in curr_tickers
            )

            sell_cost  = 0.0
            rebal_cost = 0.0
            fee_drag   = buy_cost
            turnover   = 1.0

        # ── Weekly return from current rebalance date to next week ────

        if ret_current_to_next is not None:
            weighted_ret = 0.0
            used_weight  = 0.0

            for ticker, weight in curr_weights.items():
                r = ret_current_to_next.get(ticker, np.nan)

                if not pd.isna(r):
                    weighted_ret += weight * r
                    used_weight  += weight

            gross_ret = weighted_ret / used_weight if used_weight > 0 else np.nan
            net_ret   = gross_ret - fee_drag if not pd.isna(gross_ret) else np.nan

        else:
            gross_ret = np.nan
            net_ret   = np.nan

        # ── Save portfolio-level weekly summary ──────────────────────

        all_week_rows.append({
            "date"       : date,
            "period_end" : next_date,
            "portfolio"  : pf,
            "n_stocks"   : len(curr_tickers),
            "n_entering" : len(entering),
            "n_exiting"  : len(exiting),
            "turnover"   : turnover,
            "buy_cost"   : buy_cost,
            "sell_cost"  : sell_cost,
            "rebal_cost" : rebal_cost,
            "fee_drag"   : fee_drag,
            "gross_ret"  : gross_ret,
            "net_ret"    : net_ret,
            "total_mcap" : total_mcap,
        })

        # ── Save stock-level assignments with weights ────────────────

        pf_df["date"]       = date
        pf_df["period_end"] = next_date

        pf_df["status"] = pf_df["ticker"].apply(
            lambda t: "NEW" if t in entering else "HOLD"
        )

        all_stock_rows.append(
            pf_df[[
                "date", "period_end",
                "ticker", "sector",
                "portfolio", "mom_signal",
                "within_sector_rank", "within_sector_size",
                "mcap", "mcap_date",
                "sector_weight_target",
                "weight_within_sector",
                "weight",
                "status"
            ]]
        )

        # Update previous weights for this portfolio
        prev_weights_by_pf[pf] = curr_weights
        prev_date_by_pf[pf]    = date


# ── Build final output DataFrames ────────────────────────────────────

mom_mcap_results = pd.DataFrame(all_week_rows)

mom_mcap_stock_assign = pd.concat(
    all_stock_rows,
    ignore_index=True
)

mom_mcap_stock_assign = mom_mcap_stock_assign.sort_values(
    ["date", "portfolio", "weight"],
    ascending=[True, True, False]
).reset_index(drop=True)

print("\n✓ Momentum market-cap-weighted portfolios built")

print(f"\nWeeks computed: {mom_mcap_results['date'].nunique()}")

print(f"\n{'Portfolio':<12} {'Avg Size':>9} {'Avg Turn':>10} "
      f"{'Ann Fee':>9} {'Ann Gross':>11} {'Ann Net':>10}")
print(f"{'─'*12} {'─'*9} {'─'*10} {'─'*9} {'─'*11} {'─'*10}")

for pf in MOM_PORTFOLIOS:
    pf_df = mom_mcap_results[mom_mcap_results["portfolio"] == pf]

    print(
        f"{pf:<12} "
        f"{pf_df['n_stocks'].mean():>9.0f} "
        f"{pf_df['turnover'].mean():>9.1%} "
        f"{pf_df['fee_drag'].mean()*52:>8.2%} "
        f"{pf_df['gross_ret'].mean()*52:>10.2%} "
        f"{pf_df['net_ret'].mean()*52:>9.2%}"
    )

# Weight check
weight_check = (
    mom_mcap_stock_assign
    .groupby(["date", "portfolio"])["weight"]
    .sum()
    .reset_index()
)

print("\nLatest weight-sum check:")
print(weight_check.tail(9).to_string(index=False))

MOMENTUM MODEL — WEEKLY REBALANCING WITH MARKET CAP WEIGHTS
Running weekly momentum rebalancing with market cap weights...

✓ Momentum market-cap-weighted portfolios built

Weeks computed: 275

Portfolio     Avg Size   Avg Turn   Ann Fee   Ann Gross    Ann Net
──────────── ───────── ────────── ───────── ─────────── ──────────
P1                  95     73.1%   18.94%     19.15%     0.20%
P2                 106     68.5%   17.74%     10.53%    -7.21%
P3                  95     77.6%   20.12%     10.54%    -9.60%

Latest weight-sum check:
      date portfolio  weight
2026-05-07        P1     1.0
2026-05-07        P2     1.0
2026-05-07        P3     1.0
2026-05-14        P1     1.0
2026-05-14        P2     1.0
2026-05-14        P3     1.0
2026-05-21        P1     1.0
2026-05-21        P2     1.0
2026-05-21        P3     1.0


In [70]:
# ── CELL 15: MOMENTUM MODEL — LATEST WEEK PORTFOLIO WITH WEIGHTS ─────

latest_week = mom_mcap_stock_assign["date"].max()

latest_mom = mom_mcap_stock_assign[
    mom_mcap_stock_assign["date"] == latest_week
].copy()

def fmt_rank(x):
    if pd.isna(x):
        return "-"
    return str(int(x))

def fmt_mcap_date(x):
    if pd.isna(x):
        return "-"
    return pd.to_datetime(x).date()

print("=" * 90)
print(f"MOMENTUM MODEL — MARKET-CAP-WEIGHTED PORTFOLIOS as of {latest_week.date()}")
print("=" * 90)

for pf, label in [
    ("P1", "STRONG MOMENTUM"),
    ("P2", "MIDDLE"),
    ("P3", "WEAK MOMENTUM"),
]:
    pf_df = latest_mom[
        latest_mom["portfolio"] == pf
    ].sort_values("weight", ascending=False)

    print(f"\n{'─' * 90}")
    print(f"{pf}: {label} | {len(pf_df)} stocks")
    print(f"{'─' * 90}")

    print(
        f"{'Ticker':<8} "
        f"{'Sector':<22} "
        f"{'Weight':>8} "
        f"{'Mcap (B)':>12} "
        f"{'Mcap Date':<12} "
        f"{'Mom Ret':>10} "
        f"{'Rank':>6} "
        f"{'Status':<6}"
    )

    print(
        f"{'─'*8} "
        f"{'─'*22} "
        f"{'─'*8} "
        f"{'─'*12} "
        f"{'─'*12} "
        f"{'─'*10} "
        f"{'─'*6} "
        f"{'─'*6}"
    )

    for _, r in pf_df.head(20).iterrows():
        print(
            f"{r['ticker']:<8} "
            f"{str(r['sector']):<22} "
            f"{r['weight']:>7.2%} "
            f"{r['mcap']/1e9:>11.0f} "
            f"{str(fmt_mcap_date(r['mcap_date'])):<12} "
            f"{r['mom_signal']:>9.2%} "
            f"{fmt_rank(r['within_sector_rank']):>6} "
            f"{str(r['status']):<6}"
        )

    if len(pf_df) > 20:
        print(f"... and {len(pf_df) - 20} more stocks")

    print(f"\nTotal weight      : {pf_df['weight'].sum():.2%}")
    print(f"Total market cap  : {pf_df['mcap'].sum()/1e12:.2f} trillion VND")
    print(f"Top 5 weight      : {pf_df['weight'].head(5).sum():.1%}")
    print(f"Top 10 weight     : {pf_df['weight'].head(10).sum():.1%}")

MOMENTUM MODEL — MARKET-CAP-WEIGHTED PORTFOLIOS as of 2026-05-21

──────────────────────────────────────────────────────────────────────────────────────────
P1: STRONG MOMENTUM | 95 stocks
──────────────────────────────────────────────────────────────────────────────────────────
Ticker   Sector                   Weight     Mcap (B) Mcap Date       Mom Ret   Rank Status
──────── ────────────────────── ──────── ──────────── ──────────── ────────── ────── ──────
VCB      Banks                   11.93%      500505 2026-05-12       3.61%      2 HOLD  
BID      Banks                    7.24%      303943 2026-05-12       4.02%      1 HOLD  
SSI      Financials               6.57%       69875 2026-05-12       0.18%     21 NEW   
CTG      Banks                    6.53%      273785 2026-05-12       0.97%      3 NEW   
VPL      Consumer Services        5.36%      160500 2026-05-12       0.67%      3 NEW   
BCM      Financials               5.07%       53924 2026-05-12       5.24%      2 NEW   
BV

In [71]:
# ── CELL 16: SAVE MOMENTUM MARKET-CAP-WEIGHTED OUTPUTS ───────────────

print("Saving momentum market-cap-weighted files...")

mom_mcap_results.to_csv(
    "momentum_model_weekly_market_cap_weighted_results.csv",
    index=False
)

mom_mcap_stock_assign.to_csv(
    "momentum_model_weekly_market_cap_weighted_stock_assignments.csv",
    index=False
)

files.download("momentum_model_weekly_market_cap_weighted_results.csv")
files.download("momentum_model_weekly_market_cap_weighted_stock_assignments.csv")

for pf in ["P1", "P2", "P3"]:
    fname = f"momentum_model_weekly_market_cap_weighted_{pf.lower()}.csv"

    mom_mcap_stock_assign[
        mom_mcap_stock_assign["portfolio"] == pf
    ].to_csv(fname, index=False)

    files.download(fname)

print("✓ Momentum market-cap-weighted files downloaded")

Saving momentum market-cap-weighted files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Momentum market-cap-weighted files downloaded


In [72]:
# Check what day of week the weekly prices use
print("Day of week for weekly price index:")
print("(0=Monday, 1=Tuesday, ..., 4=Friday)")
day_counts = pd.Series(weekly_prices.index.dayofweek).value_counts().sort_index()
for day, count in day_counts.items():
    day_name = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][day]
    bar = "█" * int(count / day_counts.max() * 30)
    print(f"  {day_name}: {count:>4} weeks  {bar}")

Day of week for weekly price index:
(0=Monday, 1=Tuesday, ..., 4=Friday)
  Thu:  556 weeks  ██████████████████████████████


In [73]:
# ── CELL 1: BUILD WEEKLY VALUE SIGNAL MATRIX (FORWARD-FILLED) ─────────
"""
For each weekly date, find the most recently published quarterly
value signal for each stock whose safe_date <= that weekly date.

This is forward-filling: Q1.2023 data (safe from May 15) is used
for every week from May 15 until Q2.2023 data becomes available.

Example:
  Week of May 16, 2023  → use Q1.2023 value signal
  Week of May 23, 2023  → use Q1.2023 value signal
  ...
  Week of Aug 18, 2023  → use Q2.2023 value signal (safe from Aug 14)
"""

import pandas as pd
import numpy as np

print("Building weekly value signal matrix (forward-filling quarterly data)...")

def safe_earnings_yield(pe):
    if pd.isna(pe) or pe <= PE_MIN or pe > PE_MAX:
        return np.nan
    return 1.0 / pe

def safe_book_yield(pb):
    if pd.isna(pb) or pb <= 0:
        return np.nan
    return 1.0 / pb

# Precompute value signal for all (ticker, quarter) combinations
# so we don't recompute inside the weekly loop
fund_signals = fund.copy()
fund_signals["value_signal"] = fund_signals.apply(
    lambda r: safe_book_yield(r["pb"])
    if r["sector"] in FINANCIAL_SECTORS
    else safe_earnings_yield(r["pe"]),
    axis=1
)
fund_signals["measure"] = fund_signals["sector"].apply(
    lambda s: "1/P/B" if s in FINANCIAL_SECTORS else "1/P/E"
)

# Keep only rows with valid value signal
fund_signals = fund_signals.dropna(subset=["value_signal"])

# All tickers present in both price and fundamental data
all_tickers = sorted(
    set(fund_signals["ticker"].unique()) &
    set(weekly_prices.columns)
)
print(f"  Tickers in both datasets : {len(all_tickers)}")

# Weekly dates from price data
week_dates = weekly_prices.index

# Build value signal matrix: rows=weeks, cols=tickers
# For each week, forward-fill from most recent valid quarter
print(f"  Forward-filling value signal across {len(week_dates)} weeks...")

value_weekly = pd.DataFrame(
    np.nan,
    index   = week_dates,
    columns = all_tickers
)
value_measure = pd.DataFrame(
    "",
    index   = week_dates,
    columns = all_tickers
)
value_sector = pd.DataFrame(
    "",
    index   = week_dates,
    columns = all_tickers
)

for ticker in all_tickers:
    ticker_data = (
        fund_signals[fund_signals["ticker"] == ticker]
        [["safe_date", "value_signal", "measure", "sector"]]
        .sort_values("safe_date")
    )
    if ticker_data.empty:
        continue

    for week in week_dates:
        available = ticker_data[ticker_data["safe_date"] <= week]
        if not available.empty:
            latest = available.iloc[-1]
            value_weekly.loc[week, ticker]  = latest["value_signal"]
            value_measure.loc[week, ticker] = latest["measure"]
            value_sector.loc[week, ticker]  = latest["sector"]

# Check coverage
latest_week_coverage = value_weekly.iloc[-1].notna().sum()
first_valid_week     = value_weekly.notna().any(axis=1).idxmax()

print(f"  First week with value data : {first_valid_week.date()}")
print(f"  Latest week coverage       : {latest_week_coverage} stocks")
print("✓ Weekly value signal matrix built")

Building weekly value signal matrix (forward-filling quarterly data)...
  Tickers in both datasets : 301
  Forward-filling value signal across 556 weeks...
  First week with value data : 2021-05-20
  Latest week coverage       : 301 stocks
✓ Weekly value signal matrix built


In [74]:
# ── CELL 2 (Z-SCORE VERSION): COMBINED WITHIN-SECTOR FUNCTION ────────

def assign_combo_within_sector(
    df,
    value_col,
    mom_col,
    sector_col,
    ticker_col       = "ticker",
    min_sector_size  = MIN_SECTOR_SIZE,
    zscore_clip      = 3.0,    # clip extreme z-scores at ±3
                               # prevents one outlier stock from
                               # dominating the entire sector ranking
):
    """
    Assigns each stock to P1/P2/P3 based on combo z-score
    computed within each sector.

    Steps per sector:
      1. Compute z-score of value_signal within sector
         z = (signal - sector_mean) / sector_std
      2. Compute z-score of mom_signal within sector
      3. Clip both z-scores at ±zscore_clip to handle outliers
      4. combo = 0.5 × value_zscore + 0.5 × mom_zscore
      5. Top floor(n/3)    → P1
         Bottom floor(n/3) → P3
         Middle            → P2
    """
    result = df.copy()
    result["portfolio"]          = "P2"
    result["value_zscore"]       = np.nan
    result["mom_zscore"]         = np.nan
    result["combo_score"]        = np.nan
    result["within_sector_rank"] = np.nan
    result["within_sector_size"] = np.nan

    for sector, group in result.groupby(sector_col):

        # Only stocks with BOTH signals valid
        valid_idx = group[
            group[value_col].notna() &
            group[mom_col].notna()
        ].index

        n = len(valid_idx)
        result.loc[group.index, "within_sector_size"] = n

        if n < min_sector_size:
            continue

        # ── Step 1: Value z-score within sector ──────────────────────
        val_series  = result.loc[valid_idx, value_col]
        val_mean    = val_series.mean()
        val_std     = val_series.std()

        if val_std == 0 or pd.isna(val_std):
            # All stocks have same value signal → z-score = 0 for all
            val_z = pd.Series(0.0, index=valid_idx)
        else:
            val_z = (val_series - val_mean) / val_std
            val_z = val_z.clip(-zscore_clip, zscore_clip)

        # ── Step 2: Momentum z-score within sector ────────────────────
        mom_series  = result.loc[valid_idx, mom_col]
        mom_mean    = mom_series.mean()
        mom_std     = mom_series.std()

        if mom_std == 0 or pd.isna(mom_std):
            mom_z = pd.Series(0.0, index=valid_idx)
        else:
            mom_z = (mom_series - mom_mean) / mom_std
            mom_z = mom_z.clip(-zscore_clip, zscore_clip)

        result.loc[valid_idx, "value_zscore"] = val_z
        result.loc[valid_idx, "mom_zscore"]   = mom_z

        # ── Step 3: Combo score (50/50) ───────────────────────────────
        combo = 0.5 * val_z + 0.5 * mom_z
        result.loc[valid_idx, "combo_score"] = combo

        # ── Step 4: Rank by combo score within sector ─────────────────
        combo_rank = combo.rank(ascending=False, method="first")
        result.loc[valid_idx, "within_sector_rank"] = combo_rank

        # ── Step 5: Assign P1/P2/P3 ───────────────────────────────────
        n_top = int(np.floor(n / 3))
        n_bot = int(np.floor(n / 3))

        for idx, rank in combo_rank.items():
            if rank <= n_top:
                result.loc[idx, "portfolio"] = "P1"
            elif rank > n - n_bot:
                result.loc[idx, "portfolio"] = "P3"
            else:
                result.loc[idx, "portfolio"] = "P2"

    return result

print("✓ Z-score ranking function loaded")

✓ Z-score ranking function loaded


In [75]:
# ── CELL 3: WEEKLY REBALANCING — COMBINED MODEL ───────────────────────

print("Running combined model weekly rebalancing...")

all_week_rows  = []   # weekly summary (P1 fee, return)
all_stock_rows = []   # stock-level assignments per week

prev_p1 = set()       # track P1 holdings from previous week

for i, date in enumerate(week_dates):

    # Get momentum signal for this week
    mom_row = weekly_mom.loc[date].dropna()
    if len(mom_row) < 20:
        continue

    # Get value signal for this week (forward-filled)
    val_row = value_weekly.loc[date]

    # Build combined universe DataFrame
    week_df = pd.DataFrame({
        "ticker"        : mom_row.index,
        "mom_signal"    : mom_row.values,
        "value_signal"  : [val_row.get(t, np.nan) for t in mom_row.index],
        "sector"        : [value_sector.loc[date, t]
                          if t in value_sector.columns and value_sector.loc[date, t] != ""
                          else sector_live.get(t, "Unknown")
                          for t in mom_row.index],
        "measure"       : [value_measure.loc[date, t]
                          if t in value_measure.columns
                          else ""
                          for t in mom_row.index],
    })

    # Drop stocks missing either signal
    week_df = week_df.dropna(subset=["mom_signal", "value_signal"])

    if len(week_df) < 20:
        continue

    # Within-sector combo assignment
    assigned = assign_combo_within_sector(
        df             = week_df,
        value_col      = "value_signal",
        mom_col        = "mom_signal",
        sector_col     = "sector",
        ticker_col     = "ticker",
        min_sector_size = MIN_SECTOR_SIZE,
    )

    # Current P1 holdings
    curr_p1 = set(assigned[assigned["portfolio"] == "P1"]["ticker"])

    # Compute P1 fee drag
    if i > 0:
        p1_entering = curr_p1 - prev_p1
        p1_exiting  = prev_p1 - curr_p1
        p1_size     = max(len(curr_p1), 1)

        buy_cost    = (len(p1_entering) / p1_size) * (BUY_FEE_PCT  / 100)
        sell_cost   = (len(p1_exiting)  / p1_size) * ((SELL_FEE_PCT + SELL_TAX_PCT) / 100)
        fee_drag    = buy_cost + sell_cost
        turnover    = (len(p1_entering) + len(p1_exiting)) / (2 * p1_size)
    else:
        p1_entering = curr_p1
        p1_exiting  = set()
        buy_cost    = len(curr_p1) / max(len(curr_p1), 1) * (BUY_FEE_PCT / 100)
        sell_cost   = 0.0
        fee_drag    = buy_cost
        turnover    = 1.0

    # Compute P1 gross return
    if i > 0 and date in weekly_returns.index:
        week_ret  = weekly_returns.loc[date]
        p1_rets   = [
            week_ret.get(t, np.nan) for t in curr_p1
            if t in week_ret.index and not np.isnan(week_ret[t])
        ]
        gross_ret = np.mean(p1_rets) if p1_rets else 0.0
        net_ret   = gross_ret - fee_drag
    else:
        gross_ret = net_ret = 0.0

    # Save weekly summary
    all_week_rows.append({
        "date"        : date,
        "n_stocks"    : len(curr_p1),
        "n_entering"  : len(p1_entering),
        "n_exiting"   : len(p1_exiting),
        "turnover"    : turnover,
        "buy_cost"    : buy_cost,
        "sell_cost"   : sell_cost,
        "fee_drag"    : fee_drag,
        "gross_ret"   : gross_ret,
        "net_ret"     : net_ret,
    })

    # Save stock-level assignments
    assigned["date"]   = date
    assigned["status"] = assigned["ticker"].apply(
        lambda t: "NEW"  if t in p1_entering else
                  "EXIT" if t in p1_exiting  else
                  "HOLD"
    )
    all_stock_rows.append(
    assigned[[
        "ticker", "sector", "measure",
        "value_signal", "mom_signal",
        "value_zscore", "mom_zscore", "combo_score",  # ← updated names
        "within_sector_rank", "within_sector_size",
        "portfolio", "status", "date"
    ]]
)

    prev_p1 = curr_p1

# Build output DataFrames
combo_results      = pd.DataFrame(all_week_rows)
combo_stock_assign = pd.concat(all_stock_rows, ignore_index=True)
combo_stock_assign = combo_stock_assign.sort_values(
    ["date", "portfolio", "within_sector_rank"]
).reset_index(drop=True)

print(f"\n  Weeks computed : {combo_results['date'].nunique()}")
print(f"  Avg P1 size    : {combo_results['n_stocks'].mean():.0f} stocks")
print(f"\n  P1 Performance Summary:")
print(f"  Avg weekly turnover  : {combo_results['turnover'].mean():.1%}")
print(f"  Annualized fee drag  : {combo_results['fee_drag'].mean()*52:.2%}")
print(f"  Annualized gross ret : {combo_results['gross_ret'].mean()*52:.2%}")
print(f"  Annualized net ret   : {combo_results['net_ret'].mean()*52:.2%}")
print("✓ Combined model built")


Running combined model weekly rebalancing...

  Weeks computed : 260
  Avg P1 size    : 94 stocks

  P1 Performance Summary:
  Avg weekly turnover  : 44.0%
  Annualized fee drag  : 11.43%
  Annualized gross ret : 176.57%
  Annualized net ret   : 165.14%
✓ Combined model built


In [76]:
# ── CELL 4: VALUE-MOMENTUM CORRELATION CHECK ──────────────────────────
"""
Key diagnostic from the paper:
  Value and momentum should be NEGATIVELY correlated (-0.50 to -0.60)
  This is what makes the combo powerful — they diversify each other.
"""

print("Checking value vs momentum rank correlation...")

monthly_corr = (
    combo_stock_assign
    .dropna(subset=["value_zscore", "mom_zscore"])
    .groupby("date")
    .apply(lambda x: x["value_zscore"].corr(x["mom_zscore"]))
)

print(f"\n  Value vs Momentum rank correlation:")
print(f"  Mean : {monthly_corr.mean():+.3f}")
print(f"  Std  : {monthly_corr.std():.3f}")
print(f"  Min  : {monthly_corr.min():+.3f}")
print(f"  Max  : {monthly_corr.max():+.3f}")
print(f"\n  Paper benchmark: -0.50 to -0.60")
print(f"  Negative = signals diversify each other = combo is efficient")
print(f"  Near zero = signals independent = combo still valid but less powerful")

Checking value vs momentum rank correlation...

  Value vs Momentum rank correlation:
  Mean : -0.028
  Std  : 0.092
  Min  : -0.346
  Max  : +0.236

  Paper benchmark: -0.50 to -0.60
  Negative = signals diversify each other = combo is efficient
  Near zero = signals independent = combo still valid but less powerful


In [77]:
# ── CELL 5 (UPDATED FOR Z-SCORE): LATEST WEEK PREVIEW ────────────────

latest_week   = combo_stock_assign["date"].max()
latest_combo  = combo_stock_assign[combo_stock_assign["date"] == latest_week]

print("=" * 75)
print(f"  COMBINED MODEL — as of {latest_week.date()}")
print(f"  Universe: {len(latest_combo)} stocks")
print("=" * 75)

for pf, label in [
    ("P1", "BUY   — high value AND strong momentum"),
    ("P2", "HOLD  — middle"),
    ("P3", "AVOID — low value AND weak momentum"),
]:
    pf_df = latest_combo[latest_combo["portfolio"] == pf].sort_values(
        ["sector", "within_sector_rank"]
    )

    print(f"\n  {pf}: {label} | {len(pf_df)} stocks")
    print(f"  {'Ticker':<8} {'Sector':<20} {'Measure':<7} "
          f"{'Val Z':>7} {'Mom Z':>7} {'Combo':>7} "
          f"{'WS Rank':>8} {'Status':<6}")
    print(f"  {'─'*8} {'─'*20} {'─'*7} "
          f"{'─'*7} {'─'*7} {'─'*7} "
          f"{'─'*8} {'─'*6}")

    for _, r in pf_df.head(15).iterrows():
        print(
            f"  {r['ticker']:<8} "
            f"{str(r['sector']):<20} "
            f"{str(r.get('measure','')):<7} "
            f"{r['value_zscore']:>+7.2f} "   # ← z-score: show sign (+/-)
            f"{r['mom_zscore']:>+7.2f} "      # ← z-score: show sign (+/-)
            f"{r['combo_score']:>+7.2f} "     # ← combo: show sign (+/-)
            f"{int(r['within_sector_rank']):>8} "
            f"{str(r.get('status','')):<6}"
        )
    if len(pf_df) > 15:
        print(f"  ... and {len(pf_df)-15} more stocks")

  COMBINED MODEL — as of 2026-05-21
  Universe: 298 stocks

  P1: BUY   — high value AND strong momentum | 95 stocks
  Ticker   Sector               Measure   Val Z   Mom Z   Combo  WS Rank Status
  ──────── ──────────────────── ─────── ─────── ─────── ─────── ──────── ──────
  BID      Banks                1/P/B     -0.70   +2.35   +0.82        1 HOLD  
  MSB      Banks                1/P/B     +1.23   +0.34   +0.79        2 HOLD  
  OCB      Banks                1/P/B     +1.76   -0.34   +0.71        3 HOLD  
  ACB      Banks                1/P/B     +0.35   +0.62   +0.49        4 NEW   
  VIB      Banks                1/P/B     +0.62   +0.31   +0.47        5 NEW   
  VCB      Banks                1/P/B     -1.30   +2.12   +0.41        6 NEW   
  GVR      Basic Materials      1/P/E     -0.84   +3.00   +1.08        1 NEW   
  PTB      Basic Materials      1/P/E     +0.92   +1.01   +0.96        2 HOLD  
  HII      Basic Materials      1/P/E     +2.78   -0.85   +0.96        3 HOLD  
  D

In [78]:
# ── CELL 6: SAVE AND DOWNLOAD ─────────────────────────────────────────

print("Saving combined model files...")

combo_results.to_csv("combo_model_p1_weekly_summary.csv", index=False)
combo_stock_assign.to_csv("combo_model_stock_assignments.csv", index=False)

for pf in ["P1", "P2", "P3"]:
    fname = f"combo_model_{pf.lower()}.csv"
    combo_stock_assign[combo_stock_assign["portfolio"] == pf].to_csv(
        fname, index=False
    )
    files.download(fname)

files.download("combo_model_p1_weekly_summary.csv")
files.download("combo_model_stock_assignments.csv")

print("\n" + "=" * 60)
print("✓ COMBINED MODEL COMPLETE")
print("=" * 60)
print("""
Files downloaded:

  combo_model_p1_weekly_summary.csv
    Weekly P1 performance: gross return, net return,
    fee drag, turnover, entries, exits

  combo_model_stock_assignments.csv
    Every stock, every week, with:
      value_rank, mom_rank, combo_score, portfolio, status

  combo_model_p1.csv   → BUY list history
  combo_model_p2.csv   → Middle list history
  combo_model_p3.csv   → AVOID list history

Key column: status
  NEW  = stock entered P1 this week  → you bought it
  EXIT = stock left P1 this week     → you sold it
  HOLD = stock stayed in P1          → no trade
""")

Saving combined model files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ COMBINED MODEL COMPLETE

Files downloaded:

  combo_model_p1_weekly_summary.csv
    Weekly P1 performance: gross return, net return,
    fee drag, turnover, entries, exits

  combo_model_stock_assignments.csv
    Every stock, every week, with:
      value_rank, mom_rank, combo_score, portfolio, status

  combo_model_p1.csv   → BUY list history
  combo_model_p2.csv   → Middle list history
  combo_model_p3.csv   → AVOID list history

Key column: status
  NEW  = stock entered P1 this week  → you bought it
  EXIT = stock left P1 this week     → you sold it
  HOLD = stock stayed in P1          → no trade



In [79]:
# ── CELL 5: WEEKLY REBALANCING WITH MARKET CAP WEIGHTING ─────────────
"""
Key change from previous Cell 3:
  Instead of equal weighting (simple mean of returns), we now use
  market cap weights:

      weight_i  = mcap_i / sum(mcap of all stocks in portfolio)
      gross_ret = sum(weight_i × return_i)

For fee calculation, the trade cost is now also weighted:
  When stock i enters P1, the buy fee is applied to its weight:
      buy_cost = sum(weight_i × BUY_FEE for i in entering stocks)
  When stock j exits P1, the sell fee is applied to its previous weight:
      sell_cost = sum(prev_weight_j × SELL_FEE for j in exiting stocks)
"""

print("Running weekly rebalancing with market cap weights...")

all_week_rows  = []
all_stock_rows = []

prev_p1_weights = {}   # {ticker: weight} from last week

for i, date in enumerate(week_dates):

    # Get momentum signal
    mom_row = weekly_mom.loc[date].dropna()
    if len(mom_row) < 20:
        continue

    # Get value signal (forward-filled from quarterly)
    val_row = value_weekly.loc[date]

    # Get market cap
    mcap_row = mcap_weekly.loc[date]

    # Build universe DataFrame
    week_df = pd.DataFrame({
        "ticker"       : mom_row.index,
        "mom_signal"   : mom_row.values,
        "value_signal" : [val_row.get(t, np.nan) for t in mom_row.index],
        "mcap"         : [mcap_row.get(t, np.nan) for t in mom_row.index],
        "sector"       : [value_sector.loc[date, t]
                         if t in value_sector.columns and value_sector.loc[date, t] != ""
                         else sector_live.get(t, "Unknown")
                         for t in mom_row.index],
        "measure"      : [value_measure.loc[date, t]
                         if t in value_measure.columns
                         else ""
                         for t in mom_row.index],
    })

    # Require all three: value, momentum, AND market cap
    week_df = week_df.dropna(subset=["mom_signal", "value_signal", "mcap"])
    week_df = week_df[week_df["mcap"] > 0]

    if len(week_df) < 20:
        continue

    # Z-score combo assignment
    assigned = assign_combo_within_sector(
        df              = week_df,
        value_col       = "value_signal",
        mom_col         = "mom_signal",
        sector_col      = "sector",
        ticker_col      = "ticker",
        min_sector_size = MIN_SECTOR_SIZE,
    )

    # ── Compute sector-replicated market cap weights for P1 ───────────
    p1_df = assigned[assigned["portfolio"] == "P1"].copy()

    if p1_df.empty:
        continue

    # Keep this variable because it is used later in all_week_rows
    total_mcap_p1 = p1_df["mcap"].sum()

    # 1) Sector weights from the full weekly eligible universe
    universe_sector_mcap = assigned.groupby("sector")["mcap"].sum()
    universe_total_mcap  = universe_sector_mcap.sum()

    sector_weight_universe = universe_sector_mcap / universe_total_mcap

    # 2) Market-cap weights within each P1 sector
    p1_sector_mcap = p1_df.groupby("sector")["mcap"].sum()

    # Only sectors that actually have P1 stocks can receive weight
    represented_sectors = p1_sector_mcap[p1_sector_mcap > 0].index

    # Take universe sector weights only for sectors represented in P1
    sector_weight_target = sector_weight_universe.reindex(
        represented_sectors
    ).fillna(0)

    # Redistribute unavailable sector weights across represented P1 sectors
    if sector_weight_target.sum() > 0:
        sector_weight_target = sector_weight_target / sector_weight_target.sum()
    else:
        continue

    # Map sector target weights and P1 sector market caps
    p1_df["sector_weight_target"] = p1_df["sector"].map(sector_weight_target)
    p1_df["p1_sector_mcap"]       = p1_df["sector"].map(p1_sector_mcap)

    # Stock weight inside its own sector
    p1_df["weight_within_sector"] = p1_df["mcap"] / p1_df["p1_sector_mcap"]

    # Final stock weight
    p1_df["weight"] = (
        p1_df["sector_weight_target"] *
        p1_df["weight_within_sector"]
    )

    # Clean and normalize for numerical safety
    p1_df["weight"] = p1_df["weight"].replace([np.inf, -np.inf], np.nan).fillna(0)

    if p1_df["weight"].sum() > 0:
        p1_df["weight"] = p1_df["weight"] / p1_df["weight"].sum()
    else:
        continue

    curr_p1_weights = dict(zip(p1_df["ticker"], p1_df["weight"]))
    curr_p1_tickers = set(curr_p1_weights.keys())
    prev_p1_tickers = set(prev_p1_weights.keys())
        # ── Compute weighted fee drag for P1 ──────────────────────────────
    #
    # Final convention:
    #   1. Exiting stocks:
    #        sell cost uses current rebalance date's pre-rebalance portfolio weight
    #        = drifted weight
    #
    #   2. Entering stocks:
    #        buy cost uses current rebalance date's target weight
    #
    #   3. Holding stocks:
    #        cost applies only to active rebalance trade
    #        = current target weight - drifted pre-rebalance weight

    if i > 0:
        p1_entering = curr_p1_tickers - prev_p1_tickers
        p1_exiting  = prev_p1_tickers - curr_p1_tickers
        p1_holding  = curr_p1_tickers & prev_p1_tickers

        # ------------------------------------------------------------
        # Step 1: Compute drifted weights before current rebalance
        # ------------------------------------------------------------
        #
        # drifted weight = previous weight grown by holding-period return,
        # then normalized across the previous P1 portfolio.
        #
        # Assumption:
        #   weekly_returns.loc[date] is the return from previous rebalance
        #   date to current rebalance date.

        if date in weekly_returns.index:
            week_ret_for_fee = weekly_returns.loc[date]
        else:
            week_ret_for_fee = pd.Series(dtype=float)

        drifted_values = {}

        for t, prev_w in prev_p1_weights.items():
            r = week_ret_for_fee.get(t, 0.0)

            if pd.isna(r):
                r = 0.0

            drifted_values[t] = prev_w * (1.0 + r)

        total_drifted_value = sum(drifted_values.values())

        if total_drifted_value > 0:
            drifted_p1_weights = {
                t: v / total_drifted_value
                for t, v in drifted_values.items()
            }
        else:
            drifted_p1_weights = {
                t: 0.0
                for t in prev_p1_weights.keys()
            }

        # ------------------------------------------------------------
        # Step 2: Entering stocks
        # ------------------------------------------------------------
        #
        # Entering stocks had zero previous portfolio weight.
        # Therefore buy amount = current target weight.

        buy_cost = sum(
            curr_p1_weights[t] * (BUY_FEE_PCT / 100)
            for t in p1_entering
        )

        # ------------------------------------------------------------
        # Step 3: Exiting stocks
        # ------------------------------------------------------------
        #
        # Exiting stocks are sold at their current pre-rebalance weight.
        # That is the drifted weight, not last week's target weight.

        sell_cost = sum(
            drifted_p1_weights.get(t, 0.0) * ((SELL_FEE_PCT + SELL_TAX_PCT) / 100)
            for t in p1_exiting
        )

        # ------------------------------------------------------------
        # Step 4: Holding stocks
        # ------------------------------------------------------------
        #
        # Holding stocks remain in P1.
        # We charge fees only on the active trade:
        #
        #   active_trade_w = current target weight - drifted weight
        #
        # Positive active_trade_w = buy more.
        # Negative active_trade_w = sell some.

        rebal_cost = 0.0

        for t in p1_holding:
            target_w  = curr_p1_weights.get(t, 0.0)
            drifted_w = drifted_p1_weights.get(t, 0.0)

            active_trade_w = target_w - drifted_w

            if active_trade_w > 0:
                rebal_cost += active_trade_w * (BUY_FEE_PCT / 100)

            elif active_trade_w < 0:
                rebal_cost += abs(active_trade_w) * (
                    (SELL_FEE_PCT + SELL_TAX_PCT) / 100
                )

        fee_drag = buy_cost + sell_cost + rebal_cost

        # ------------------------------------------------------------
        # Turnover
        # ------------------------------------------------------------
        #
        # Turnover is based on actual active traded weight.
        # Entering: current target weight.
        # Exiting : drifted current pre-rebalance weight.
        # Holding : absolute active adjustment.

        entering_turnover = sum(
            curr_p1_weights[t]
            for t in p1_entering
        )

        exiting_turnover = sum(
            drifted_p1_weights.get(t, 0.0)
            for t in p1_exiting
        )

        holding_turnover = 0.0

        for t in p1_holding:
            target_w  = curr_p1_weights.get(t, 0.0)
            drifted_w = drifted_p1_weights.get(t, 0.0)

            holding_turnover += abs(target_w - drifted_w)

        turnover = (
            entering_turnover +
            exiting_turnover +
            holding_turnover
        ) / 2

    else:
        # First week — buy all P1 stocks
        p1_entering = curr_p1_tickers
        p1_exiting  = set()

        buy_cost = sum(
            curr_p1_weights[t] * (BUY_FEE_PCT / 100)
            for t in curr_p1_tickers
        )

        sell_cost  = 0.0
        rebal_cost = 0.0
        fee_drag   = buy_cost
        turnover   = 1.0

    # ── Compute weighted gross return for P1 ──────────────────────────
    if i > 0 and date in weekly_returns.index:
        week_ret = weekly_returns.loc[date]

        weighted_ret = 0.0
        used_weight  = 0.0
        for ticker, weight in curr_p1_weights.items():
            if ticker in week_ret.index and not np.isnan(week_ret[ticker]):
                weighted_ret += weight * week_ret[ticker]
                used_weight  += weight

        # Normalize if some stocks had NaN returns
        gross_ret = weighted_ret / used_weight if used_weight > 0 else 0.0
        net_ret   = gross_ret - fee_drag
    else:
        gross_ret = net_ret = 0.0

    # ── Save weekly summary ───────────────────────────────────────────
    all_week_rows.append({
        "date"       : date,
        "n_stocks"   : len(curr_p1_tickers),
        "n_entering" : len(p1_entering),
        "n_exiting"  : len(p1_exiting),
        "turnover"   : turnover,
        "buy_cost"   : buy_cost,
        "sell_cost"  : sell_cost,
        "rebal_cost" : rebal_cost,
        "fee_drag"   : fee_drag,
        "gross_ret"  : gross_ret,
        "net_ret"    : net_ret,
        "total_mcap" : total_mcap_p1,
    })

    # ── Save stock-level assignments with weights ─────────────────────
    assigned["date"]   = date
    assigned["weight"] = assigned["ticker"].apply(
        lambda t: curr_p1_weights.get(t, 0.0)
    )
    assigned["status"] = assigned["ticker"].apply(
        lambda t: "NEW"  if t in p1_entering else
                  "EXIT" if t in p1_exiting  else
                  "HOLD"
    )

    all_stock_rows.append(
        assigned[[
            "ticker", "sector", "measure",
            "value_signal", "mom_signal", "mcap", "weight",
            "value_zscore", "mom_zscore", "combo_score",
            "within_sector_rank", "within_sector_size",
            "portfolio", "status", "date"
        ]]
    )

    prev_p1_weights = curr_p1_weights

# Build output DataFrames
combo_results      = pd.DataFrame(all_week_rows)
combo_stock_assign = pd.concat(all_stock_rows, ignore_index=True)
combo_stock_assign = combo_stock_assign.sort_values(
    ["date", "portfolio", "within_sector_rank"]
).reset_index(drop=True)

print(f"\n  Weeks computed         : {combo_results['date'].nunique()}")
print(f"  Avg P1 size            : {combo_results['n_stocks'].mean():.0f} stocks")
print(f"\n  P1 (Market Cap Weighted) Performance Summary:")
print(f"  Avg weekly turnover     : {combo_results['turnover'].mean():.1%}")
print(f"  Annualized fee drag     : {combo_results['fee_drag'].mean()*52:.2%}")
print(f"  Annualized gross return : {combo_results['gross_ret'].mean()*52:.2%}")
print(f"  Annualized net return   : {combo_results['net_ret'].mean()*52:.2%}")
print("✓ Market cap weighted model built")


# ── CELL 6: LATEST WEEK PORTFOLIO WITH WEIGHTS ───────────────────────

latest_week  = combo_stock_assign["date"].max()
latest_combo = combo_stock_assign[combo_stock_assign["date"] == latest_week]
latest_p1    = latest_combo[latest_combo["portfolio"] == "P1"].sort_values(
    "weight", ascending=False
)

print("=" * 75)
print(f"  P1 PORTFOLIO (Market Cap Weighted) — as of {latest_week.date()}")
print(f"  {len(latest_p1)} stocks  |  Total mcap: "
      f"{latest_p1['mcap'].sum()/1e12:.2f} trillion VND")
print("=" * 75)
print(f"\n  {'Ticker':<8} {'Sector':<20} {'Weight':>7} "
      f"{'Mcap (B)':>12} {'Val Z':>7} {'Mom Z':>7} {'Combo':>7} {'Status':<6}")
print(f"  {'─'*8} {'─'*20} {'─'*7} {'─'*12} "
      f"{'─'*7} {'─'*7} {'─'*7} {'─'*6}")

for _, r in latest_p1.head(20).iterrows():
    print(
        f"  {r['ticker']:<8} "
        f"{str(r['sector']):<20} "
        f"{r['weight']:>6.2%} "
        f"{r['mcap']/1e9:>11.0f}  "
        f"{r['value_zscore']:>+7.2f} "
        f"{r['mom_zscore']:>+7.2f} "
        f"{r['combo_score']:>+7.2f} "
        f"{str(r.get('status', '')):<6}"
    )

if len(latest_p1) > 20:
    print(f"  ... and {len(latest_p1)-20} more stocks")

# Weight concentration check
top5_weight = latest_p1["weight"].head(5).sum()
top10_weight = latest_p1["weight"].head(10).sum()
print(f"\n  Top 5 stocks weight  : {top5_weight:.1%}")
print(f"  Top 10 stocks weight : {top10_weight:.1%}")
print(f"  (High concentration = portfolio dominated by few large caps)")

Running weekly rebalancing with market cap weights...

  Weeks computed         : 260
  Avg P1 size            : 94 stocks

  P1 (Market Cap Weighted) Performance Summary:
  Avg weekly turnover     : 67.0%
  Annualized fee drag     : 17.42%
  Annualized gross return : 228.05%
  Annualized net return   : 210.63%
✓ Market cap weighted model built
  P1 PORTFOLIO (Market Cap Weighted) — as of 2026-05-21
  95 stocks  |  Total mcap: 1975.60 trillion VND

  Ticker   Sector                Weight     Mcap (B)   Val Z   Mom Z   Combo Status
  ──────── ──────────────────── ─────── ──────────── ─────── ─────── ─────── ──────
  VCB      Banks                15.94%      500505    -1.30   +2.12   +0.41 NEW   
  BID      Banks                 9.68%      303943    -0.70   +2.35   +0.82 HOLD  
  BCM      Financials            9.41%       53924    -1.02   +2.53   +0.76 NEW   
  BVH      Financials            8.62%       49439    -0.86   +2.69   +0.91 NEW   
  NVL      Financials            6.35%       36

In [80]:
# ── CELL 7: SAVE AND DOWNLOAD ─────────────────────────────────────────

print("Saving market cap weighted model files...")

combo_results.to_csv("combo_mcap_weighted_summary.csv", index=False)
combo_stock_assign.to_csv("combo_mcap_weighted_stocks.csv", index=False)

# Save P1 only
combo_stock_assign[combo_stock_assign["portfolio"] == "P1"].to_csv(
    "combo_mcap_weighted_p1.csv", index=False
)

files.download("combo_mcap_weighted_summary.csv")
files.download("combo_mcap_weighted_stocks.csv")
files.download("combo_mcap_weighted_p1.csv")

print("\n" + "=" * 60)
print("✓ MARKET CAP WEIGHTED MODEL COMPLETE")
print("=" * 60)
print("""
Files downloaded:

  combo_mcap_weighted_summary.csv
    Weekly P1 performance with mcap-weighted returns and fees

  combo_mcap_weighted_stocks.csv
    All stocks every week with weights, signals, portfolio assignment

  combo_mcap_weighted_p1.csv
    P1 holdings only (BUY list with weights and status)

Key new column: weight
  Shows each stock's portfolio weight = mcap_i / sum(mcap of P1)
  Stocks with higher market cap get larger weights
  Sum of weights within P1 = 1.0 (or 100%)
""")

Saving market cap weighted model files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ MARKET CAP WEIGHTED MODEL COMPLETE

Files downloaded:

  combo_mcap_weighted_summary.csv
    Weekly P1 performance with mcap-weighted returns and fees

  combo_mcap_weighted_stocks.csv
    All stocks every week with weights, signals, portfolio assignment

  combo_mcap_weighted_p1.csv
    P1 holdings only (BUY list with weights and status)

Key new column: weight
  Shows each stock's portfolio weight = mcap_i / sum(mcap of P1)
  Stocks with higher market cap get larger weights
  Sum of weights within P1 = 1.0 (or 100%)

